# Civitai LoRA → Draw Things → Google Drive

Configure the URL, execute the form, then press **Run conversion**. A standalone conversion workflow; no training or image generation. **Experimental Linux build: not yet tested in Colab.**


In [ ]:
#@title Civitai LoRA → Draw Things — Configure { display-mode: "form" }
# Requires: only the Python standard library.
# Produces: SUBMITTED_JOB and PENDING_JOB_PATH.
# Safe to rerun: yes; atomically replaces only the pending-job snapshot.

from datetime import datetime, timezone
import json
from pathlib import Path
import re
from urllib.parse import urlsplit
import uuid

_FORM_EMBEDDED_PROJECT_REVISION = '32562b58e1e554f3ee1817ac46631110e7f11d84097379f7086b09dd60b2788a'  # generated

#@markdown Add a Civitai page or download URL. Optional authentication uses Colab Secrets, never a token in this form.
#@markdown The official converter is built from pinned source on Linux. This Colab build path is experimental and has not been exercised end to end.
# The builder replaces this marker from JobConfig and FORM_* metadata.
#@markdown Embedded mode is ready to run. Blank source fields are unused; blank model_version uses upstream auto-detection.
#@markdown ### Project source
#@markdown Embedded is deterministic and works in a fresh runtime. Repository mode requires an immutable 40-character commit SHA. Local mode requires revision `local`.
#@markdown Project namespace: `lora-converter`
project_id = 'lora-converter'
project_source_mode = 'embedded' #@param ["embedded", "repository", "local"]
project_repository_url = '' #@param {type:"string"}
project_revision = '' #@param {type:"string"}
project_subdirectory = '' #@param {type:"string"}
project_source_path = '' #@param {type:"string"}

#@markdown ### Civitai LoRA
input_url = 'https://civitai.red/api/download/models/3404818?fileId=3296039' #@param {type:"string"}
civitai_secret_name = 'CIVITAI_API_TOKEN' #@param {type:"string"}

#@markdown ### Conversion
model_version = '' #@param {type:"string"}
scale_factor = 1.0 #@param {type:"number"}
build_jobs = 1 #@param {type:"integer"}

#@markdown ### Output
output_name = 'civitai_3404818_3296039.ckpt' #@param {type:"string"}
auto_download = True #@param {type:"boolean"}

#@markdown ### Persistence and execution
persistent_root = '/content/drive/MyDrive/production_colab/lora-converter' #@param {type:"string"}
runtime_root = '/content/production_colab/lora-converter' #@param {type:"string"}
reuse_cache = True #@param {type:"boolean"}

if not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", project_id):
    raise ValueError("project_id must use lowercase letters, numbers, and single hyphens")
_repository = urlsplit(project_repository_url) if project_repository_url else None
if _repository and (_repository.username or _repository.password or _repository.query):
    raise ValueError(
        "Do not place credentials or signed query strings in project_repository_url. "
        "Use the runtime's configured Git authentication."
    )
if project_source_mode == "repository" and not re.fullmatch(r"[0-9a-fA-F]{40}", project_revision):
    raise ValueError("Repository mode requires an immutable 40-character commit SHA")
if project_source_mode == "local" and (not project_source_path or project_revision != "local"):
    raise ValueError("Local mode requires project_source_path and project_revision='local'")

_configuration = {
    'project_id': project_id,
    'project_source_mode': project_source_mode,
    'project_repository_url': project_repository_url,
    'project_revision': project_revision,
    'project_subdirectory': project_subdirectory,
    'project_source_path': project_source_path,
    'input_url': input_url,
    'civitai_secret_name': civitai_secret_name,
    'model_version': model_version,
    'scale_factor': scale_factor,
    'build_jobs': build_jobs,
    'output_name': output_name,
    'auto_download': auto_download,
    'persistent_root': persistent_root,
    'runtime_root': runtime_root,
    'reuse_cache': reuse_cache,
}

SUBMITTED_JOB = {
    "policy_revision": "production-colab-v4",
    "submission_id": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-"
    + uuid.uuid4().hex[:8],
    "submitted_utc": datetime.now(timezone.utc).isoformat(),
    "submitted_project_revision": (
        _FORM_EMBEDDED_PROJECT_REVISION
        if project_source_mode == "embedded"
        else project_revision
    ),
    "configuration": _configuration,
}

PENDING_JOB_PATH = Path(runtime_root).expanduser() / "state" / "pending_job.json"
PENDING_JOB_PATH.parent.mkdir(parents=True, exist_ok=True)
_temporary = PENDING_JOB_PATH.with_suffix(".json.tmp")
_temporary.write_text(
    json.dumps(SUBMITTED_JOB, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
_temporary.replace(PENDING_JOB_PATH)

print("Job configured. Processing has not started.")
print(f"Frozen submission: {PENDING_JOB_PATH}")
print(f"Project source: {project_source_mode}")
print(f"Persistent root: {persistent_root}")
print(f"Final output: /content/outputs/{output_name}")


In [ ]:
#@title Run complete workflow { display-mode: "form" }
# Requires: the frozen pending-job JSON written by the form, or its persistent registry entry.
# Produces: PROJECT_ROOT, CONFIG, RUN_RESULT, and DELIVERY_RESULT.
# Safe to rerun: yes; creates a new isolated run and never depends on lower cells.

from datetime import datetime, timezone
import base64
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import sys
import time
import traceback
from urllib.parse import urlsplit, urlunsplit
import uuid

_EMBEDDED_PROJECT_FILES = {"requirements.lock": "IyBTdGFuZGFyZCBsaWJyYXJ5IGFwcGxpY2F0aW9uOyB0aGUgb2ZmaWNpYWwgU3dpZnQvQmF6ZWwgY29udmVydGVyIGhhcyBhIHNlcGFyYXRlIHBpbm5lZCB0b29sY2hhaW4uCg==", "src/project_app/__init__.py": "IiIiSW1wb3J0YWJsZSBpbXBsZW1lbnRhdGlvbiBmb3IgdGhlIHByb2R1Y3Rpb24gQ29sYWIgdGVtcGxhdGUuIiIiCgpmcm9tIC5jb25maWcgaW1wb3J0IEpvYkNvbmZpZywgbG9hZF9wZW5kaW5nX2pvYgpmcm9tIC5vcmNoZXN0cmF0b3IgaW1wb3J0IHJ1bl9jb21wbGV0ZV9qb2IKCl9fYWxsX18gPSBbIkpvYkNvbmZpZyIsICJsb2FkX3BlbmRpbmdfam9iIiwgInJ1bl9jb21wbGV0ZV9qb2IiXQo=", "src/project_app/cache.py": "IiIiU21hbGwgcmV1c2FibGUgY29udGVudC1hZGRyZXNzZWQgSlNPTiBjYWNoZSB3aXRoIGV4cGxpY2l0IHByb3ZlbmFuY2UuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpDQUNIRV9FTlZFTE9QRV9WRVJTSU9OID0gImpzb24tY2FjaGUtZW52ZWxvcGUtdjEiCgoKZGVmIGNhbm9uaWNhbF9qc29uKHBheWxvYWQ6IEFueSkgLT4gc3RyOgogICAgcmV0dXJuIGpzb24uZHVtcHMocGF5bG9hZCwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSwgZW5zdXJlX2FzY2lpPUZhbHNlKQoKCmRlZiBjYWNoZV9rZXkoc3RhZ2U6IHN0ciwgaW1wbGVtZW50YXRpb25fdmVyc2lvbjogc3RyLCBzZW1hbnRpY19pbnB1dHM6IGRpY3Rbc3RyLCBBbnldKSAtPiBzdHI6CiAgICBwYXlsb2FkID0gewogICAgICAgICJzdGFnZSI6IHN0YWdlLAogICAgICAgICJpbXBsZW1lbnRhdGlvbl92ZXJzaW9uIjogaW1wbGVtZW50YXRpb25fdmVyc2lvbiwKICAgICAgICAic2VtYW50aWNfaW5wdXRzIjogc2VtYW50aWNfaW5wdXRzLAogICAgfQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KGNhbm9uaWNhbF9qc29uKHBheWxvYWQpLmVuY29kZSgidXRmLTgiKSkuaGV4ZGlnZXN0KCkKCgpkZWYgY29udGVudF9maW5nZXJwcmludChwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ2FjaGVMb29rdXA6CiAgICBzdGF0dXM6IHN0cgogICAga2V5OiBzdHIKICAgIHZhbHVlOiBBbnkgfCBOb25lID0gTm9uZQogICAgcmVhc29uOiBzdHIgPSAiIgogICAgcGF0aDogc3RyID0gIiIKCiAgICBkZWYgZXZlbnQoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInN0YXR1cyI6IHNlbGYuc3RhdHVzLAogICAgICAgICAgICAia2V5Ijogc2VsZi5rZXksCiAgICAgICAgICAgICJyZWFzb24iOiBzZWxmLnJlYXNvbiwKICAgICAgICAgICAgInBhdGgiOiBzZWxmLnBhdGgsCiAgICAgICAgfQoKCmNsYXNzIEpzb25TdGFnZUNhY2hlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJvb3Q6IFBhdGgsIHN0YWdlOiBzdHIsIGltcGxlbWVudGF0aW9uX3ZlcnNpb246IHN0cik6CiAgICAgICAgc2VsZi5yb290ID0gcm9vdCAvIHN0YWdlCiAgICAgICAgc2VsZi5zdGFnZSA9IHN0YWdlCiAgICAgICAgc2VsZi5pbXBsZW1lbnRhdGlvbl92ZXJzaW9uID0gaW1wbGVtZW50YXRpb25fdmVyc2lvbgoKICAgIGRlZiBrZXkoc2VsZiwgc2VtYW50aWNfaW5wdXRzOiBkaWN0W3N0ciwgQW55XSkgLT4gc3RyOgogICAgICAgIHJldHVybiBjYWNoZV9rZXkoc2VsZi5zdGFnZSwgc2VsZi5pbXBsZW1lbnRhdGlvbl92ZXJzaW9uLCBzZW1hbnRpY19pbnB1dHMpCgogICAgZGVmIHBhdGhfZm9yKHNlbGYsIGtleTogc3RyKSAtPiBQYXRoOgogICAgICAgIHJldHVybiBzZWxmLnJvb3QgLyBmIntrZXl9Lmpzb24iCgogICAgZGVmIGxvYWQoc2VsZiwgc2VtYW50aWNfaW5wdXRzOiBkaWN0W3N0ciwgQW55XSkgLT4gQ2FjaGVMb29rdXA6CiAgICAgICAga2V5ID0gc2VsZi5rZXkoc2VtYW50aWNfaW5wdXRzKQogICAgICAgIHBhdGggPSBzZWxmLnBhdGhfZm9yKGtleSkKICAgICAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmV0dXJuIENhY2hlTG9va3VwKCJtaXNzIiwga2V5LCByZWFzb249Im5vdF9mb3VuZCIsIHBhdGg9c3RyKHBhdGgpKQogICAgICAgIHRyeToKICAgICAgICAgICAgZW52ZWxvcGUgPSBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICBpZiBlbnZlbG9wZS5nZXQoImVudmVsb3BlX3ZlcnNpb24iKSAhPSBDQUNIRV9FTlZFTE9QRV9WRVJTSU9OOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZW52ZWxvcGVfdmVyc2lvbl9jaGFuZ2VkIikKICAgICAgICAgICAgaWYgZW52ZWxvcGUuZ2V0KCJzdGFnZSIpICE9IHNlbGYuc3RhZ2U6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzdGFnZV9jaGFuZ2VkIikKICAgICAgICAgICAgaWYgZW52ZWxvcGUuZ2V0KCJpbXBsZW1lbnRhdGlvbl92ZXJzaW9uIikgIT0gc2VsZi5pbXBsZW1lbnRhdGlvbl92ZXJzaW9uOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW1wbGVtZW50YXRpb25fdmVyc2lvbl9jaGFuZ2VkIikKICAgICAgICAgICAgaWYgZW52ZWxvcGUuZ2V0KCJrZXkiKSAhPSBrZXk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJrZXlfbWlzbWF0Y2giKQogICAgICAgICAgICByZXR1cm4gQ2FjaGVMb29rdXAoImhpdCIsIGtleSwgdmFsdWU9ZW52ZWxvcGVbInZhbHVlIl0sIHBhdGg9c3RyKHBhdGgpKQogICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvciwgS2V5RXJyb3IsIFR5cGVFcnJvciwganNvbi5KU09ORGVjb2RlRXJyb3IpIGFzIGV4YzoKICAgICAgICAgICAgcXVhcmFudGluZSA9IHBhdGgud2l0aF9zdWZmaXgoCiAgICAgICAgICAgICAgICBwYXRoLnN1ZmZpeAogICAgICAgICAgICAgICAgKyAiLmNvcnJ1cHQtIgogICAgICAgICAgICAgICAgKyBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5zdHJmdGltZSgiJVklbSVkVCVIJU0lU1oiKQogICAgICAgICAgICApCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHBhdGgucmVwbGFjZShxdWFyYW50aW5lKQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUgPSBwYXRoCiAgICAgICAgICAgIHJldHVybiBDYWNoZUxvb2t1cCgKICAgICAgICAgICAgICAgICJtaXNzIiwKICAgICAgICAgICAgICAgIGtleSwKICAgICAgICAgICAgICAgIHJlYXNvbj1mImNvcnJ1cHQ6e3R5cGUoZXhjKS5fX25hbWVfX30iLAogICAgICAgICAgICAgICAgcGF0aD1zdHIocXVhcmFudGluZSksCiAgICAgICAgICAgICkKCiAgICBkZWYgc2F2ZShzZWxmLCBzZW1hbnRpY19pbnB1dHM6IGRpY3Rbc3RyLCBBbnldLCB2YWx1ZTogQW55KSAtPiBDYWNoZUxvb2t1cDoKICAgICAgICBrZXkgPSBzZWxmLmtleShzZW1hbnRpY19pbnB1dHMpCiAgICAgICAgcGF0aCA9IHNlbGYucGF0aF9mb3Ioa2V5KQogICAgICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBlbnZlbG9wZSA9IHsKICAgICAgICAgICAgImVudmVsb3BlX3ZlcnNpb24iOiBDQUNIRV9FTlZFTE9QRV9WRVJTSU9OLAogICAgICAgICAgICAic3RhZ2UiOiBzZWxmLnN0YWdlLAogICAgICAgICAgICAiaW1wbGVtZW50YXRpb25fdmVyc2lvbiI6IHNlbGYuaW1wbGVtZW50YXRpb25fdmVyc2lvbiwKICAgICAgICAgICAgImtleSI6IGtleSwKICAgICAgICAgICAgInNlbWFudGljX2lucHV0cyI6IHNlbWFudGljX2lucHV0cywKICAgICAgICAgICAgImNyZWF0ZWRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgICAgICJ2YWx1ZSI6IHZhbHVlLAogICAgICAgIH0KICAgICAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgICAgIHRlbXBvcmFyeS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZW52ZWxvcGUsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCiAgICAgICAgcmV0dXJuIENhY2hlTG9va3VwKCJnZW5lcmF0ZWQiLCBrZXksIHZhbHVlPXZhbHVlLCBwYXRoPXN0cihwYXRoKSkK", "src/project_app/config.py": "IiIiQ2Fub25pY2FsIGNvbmZpZ3VyYXRpb24gYW5kIGltbXV0YWJsZSBzdWJtaXR0ZWQtam9iIHNlcmlhbGl6YXRpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcwpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmltcG9ydCByZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55CmZyb20gdXJsbGliLnBhcnNlIGltcG9ydCB1cmxzcGxpdAppbXBvcnQgdXVpZAoKIyBDdXJyZW50IGNhbm9uaWNhbCBob3N0cyBhbmQgbGVnYWN5IGFsaWFzZXMgZG9jdW1lbnRlZCBieSBDaXZpdGFpJ3Mgb3duIHNvdXJjZS4KQ0lWSVRBSV9IT1NUUyA9IGZyb3plbnNldCh7ImNpdml0YWkuY29tIiwgInd3dy5jaXZpdGFpLmNvbSIsICJjaXZpdGFpLnJlZCIsICJ3d3cuY2l2aXRhaS5yZWQiLCAiY2l2aXRhaS5ncmVlbiIsICJ3d3cuY2l2aXRhaS5ncmVlbiIsICJjaXZpdGFpLmJ1enoiLCAid3d3LmNpdml0YWkuYnV6eiJ9KQoKCmRlZiBjaXZpdGFpX29yaWdpbih1cmw6IHN0cikgLT4gc3RyOgogICAgcGFyc2VkID0gdXJsc3BsaXQodXJsLnN0cmlwKCkpCiAgICBpZiBwYXJzZWQuc2NoZW1lICE9ICJodHRwcyIgb3IgcGFyc2VkLmhvc3RuYW1lIG5vdCBpbiBDSVZJVEFJX0hPU1RTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVzZSBhbiBIVFRQUyBDaXZpdGFpIFVSTCAoLmNvbSwgLnJlZCwgLmdyZWVuIG9yIC5idXp6KSIpCiAgICBpZiBwYXJzZWQudXNlcm5hbWUgb3IgcGFyc2VkLnBhc3N3b3JkIG9yIHBhcnNlZC5wb3J0IG5vdCBpbiB7Tm9uZSwgNDQzfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaXZpdGFpIFVSTHMgbXVzdCBub3QgY29udGFpbiBjcmVkZW50aWFscyBvciBhIGN1c3RvbSBwb3J0IikKICAgIGhvc3QgPSBwYXJzZWQuaG9zdG5hbWUucmVtb3ZlcHJlZml4KCJ3d3cuIikKICAgICMgTGVnYWN5IGRvbWFpbnMgbW92ZWQgdG8gLmNvbTsgcHJlc2VydmUgdGhlIC5yZWQgQVBJIGZvciAucmVkIGlucHV0cy4KICAgIHJldHVybiAiaHR0cHM6Ly8iICsgKCJjaXZpdGFpLnJlZCIgaWYgaG9zdCA9PSAiY2l2aXRhaS5yZWQiIGVsc2UgImNpdml0YWkuY29tIikKCgpQT0xJQ1lfUkVWSVNJT04gPSAicHJvZHVjdGlvbi1jb2xhYi12NCIKCiMgQ2Fub25pY2FsIHByZXNlbnRhdGlvbiBtZXRhZGF0YSBmb3IgdGhlIGdlbmVyYXRlZCBDb2xhYiBmb3JtLiBEZWZhdWx0cyBhbmQKIyBQeXRob24gdHlwZXMgbGl2ZSBvbmx5IG9uIEpvYkNvbmZpZzsgdGhlIGJ1aWxkZXIgdmVyaWZpZXMgdGhhdCBldmVyeSBmaWVsZCBpcwojIHByZXNlbnRlZCBleGFjdGx5IG9uY2UuCkZPUk1fU0VDVElPTlMgPSAoCiAgICAoCiAgICAgICAgIlByb2plY3Qgc291cmNlIiwKICAgICAgICAoCiAgICAgICAgICAgICJwcm9qZWN0X2lkIiwKICAgICAgICAgICAgInByb2plY3Rfc291cmNlX21vZGUiLAogICAgICAgICAgICAicHJvamVjdF9yZXBvc2l0b3J5X3VybCIsCiAgICAgICAgICAgICJwcm9qZWN0X3JldmlzaW9uIiwKICAgICAgICAgICAgInByb2plY3Rfc3ViZGlyZWN0b3J5IiwKICAgICAgICAgICAgInByb2plY3Rfc291cmNlX3BhdGgiLAogICAgICAgICksCiAgICApLAogICAgKCJDaXZpdGFpIExvUkEiLCAoImlucHV0X3VybCIsICJjaXZpdGFpX3NlY3JldF9uYW1lIikpLAogICAgKCJDb252ZXJzaW9uIiwgKCJtb2RlbF92ZXJzaW9uIiwgInNjYWxlX2ZhY3RvciIsICJidWlsZF9qb2JzIikpLAogICAgKCJPdXRwdXQiLCAoIm91dHB1dF9uYW1lIiwgImF1dG9fZG93bmxvYWQiKSksCiAgICAoIlBlcnNpc3RlbmNlIGFuZCBleGVjdXRpb24iLCAoInBlcnNpc3RlbnRfcm9vdCIsICJydW50aW1lX3Jvb3QiLCAicmV1c2VfY2FjaGUiKSksCikKCkZPUk1fQ09OVFJPTFMgPSB7CiAgICAicHJvamVjdF9pZCI6IE5vbmUsCiAgICAicHJvamVjdF9zb3VyY2VfbW9kZSI6ICdbImVtYmVkZGVkIiwgInJlcG9zaXRvcnkiLCAibG9jYWwiXScsCiAgICAicHJvamVjdF9yZXBvc2l0b3J5X3VybCI6ICd7dHlwZToic3RyaW5nIn0nLAogICAgInByb2plY3RfcmV2aXNpb24iOiAne3R5cGU6InN0cmluZyJ9JywKICAgICJwcm9qZWN0X3N1YmRpcmVjdG9yeSI6ICd7dHlwZToic3RyaW5nIn0nLAogICAgInByb2plY3Rfc291cmNlX3BhdGgiOiAne3R5cGU6InN0cmluZyJ9JywKICAgICJpbnB1dF91cmwiOiAne3R5cGU6InN0cmluZyJ9JywKICAgICJjaXZpdGFpX3NlY3JldF9uYW1lIjogJ3t0eXBlOiJzdHJpbmcifScsCiAgICAibW9kZWxfdmVyc2lvbiI6ICd7dHlwZToic3RyaW5nIn0nLAogICAgInNjYWxlX2ZhY3RvciI6ICd7dHlwZToibnVtYmVyIn0nLAogICAgImJ1aWxkX2pvYnMiOiAne3R5cGU6ImludGVnZXIifScsCiAgICAib3V0cHV0X25hbWUiOiAne3R5cGU6InN0cmluZyJ9JywKICAgICJhdXRvX2Rvd25sb2FkIjogJ3t0eXBlOiJib29sZWFuIn0nLAogICAgInBlcnNpc3RlbnRfcm9vdCI6ICd7dHlwZToic3RyaW5nIn0nLAogICAgInJ1bnRpbWVfcm9vdCI6ICd7dHlwZToic3RyaW5nIn0nLAogICAgInJldXNlX2NhY2hlIjogJ3t0eXBlOiJib29sZWFuIn0nLAp9CgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSm9iQ29uZmlnOgogICAgcHJvamVjdF9pZDogc3RyID0gJ2xvcmEtY29udmVydGVyJwogICAgcHJvamVjdF9zb3VyY2VfbW9kZTogc3RyID0gJ2VtYmVkZGVkJwogICAgcHJvamVjdF9yZXBvc2l0b3J5X3VybDogc3RyID0gJycKICAgIHByb2plY3RfcmV2aXNpb246IHN0ciA9ICcnCiAgICBwcm9qZWN0X3N1YmRpcmVjdG9yeTogc3RyID0gJycKICAgIHByb2plY3Rfc291cmNlX3BhdGg6IHN0ciA9ICIiCiAgICBpbnB1dF91cmw6IHN0ciA9ICJodHRwczovL2Npdml0YWkucmVkL2FwaS9kb3dubG9hZC9tb2RlbHMvMzQwNDgxOD9maWxlSWQ9MzI5NjAzOSIKICAgIGNpdml0YWlfc2VjcmV0X25hbWU6IHN0ciA9ICJDSVZJVEFJX0FQSV9UT0tFTiIKICAgIG1vZGVsX3ZlcnNpb246IHN0ciA9ICIiCiAgICBzY2FsZV9mYWN0b3I6IGZsb2F0ID0gMS4wCiAgICBidWlsZF9qb2JzOiBpbnQgPSAxCiAgICBvdXRwdXRfbmFtZTogc3RyID0gImNpdml0YWlfMzQwNDgxOF8zMjk2MDM5LmNrcHQiCiAgICBhdXRvX2Rvd25sb2FkOiBib29sID0gVHJ1ZQogICAgcGVyc2lzdGVudF9yb290OiBzdHIgPSAnL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9wcm9kdWN0aW9uX2NvbGFiL2xvcmEtY29udmVydGVyJwogICAgcnVudGltZV9yb290OiBzdHIgPSAnL2NvbnRlbnQvcHJvZHVjdGlvbl9jb2xhYi9sb3JhLWNvbnZlcnRlcicKICAgIHJldXNlX2NhY2hlOiBib29sID0gVHJ1ZQoKICAgIGRlZiB2YWxpZGF0ZShzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbYS16MC05XSsoPzotW2EtejAtOV0rKSoiLCBzZWxmLnByb2plY3RfaWQpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9qZWN0X2lkIG11c3QgdXNlIGxvd2VyY2FzZSBsZXR0ZXJzLCBudW1iZXJzLCBhbmQgc2luZ2xlIGh5cGhlbnMiKQogICAgICAgIGlmIHNlbGYucHJvamVjdF9zb3VyY2VfbW9kZSBub3QgaW4geyJlbWJlZGRlZCIsICJyZXBvc2l0b3J5IiwgImxvY2FsIn06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBwcm9qZWN0X3NvdXJjZV9tb2RlOiB7c2VsZi5wcm9qZWN0X3NvdXJjZV9tb2RlfSIpCiAgICAgICAgaWYgbm90IHNlbGYub3V0cHV0X25hbWUuc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigib3V0cHV0X25hbWUgbXVzdCBub3QgYmUgYmxhbmsiKQogICAgICAgIGlmIFBhdGgoc2VsZi5vdXRwdXRfbmFtZSkubmFtZSAhPSBzZWxmLm91dHB1dF9uYW1lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJvdXRwdXRfbmFtZSBtdXN0IGJlIGEgZmlsZW5hbWUsIG5vdCBhIG5lc3RlZCBwYXRoIikKICAgICAgICBpZiBzZWxmLmlucHV0X3VybDoKICAgICAgICAgICAgcGFyc2VkID0gdXJsc3BsaXQoc2VsZi5pbnB1dF91cmwpCiAgICAgICAgICAgIGNpdml0YWlfb3JpZ2luKHNlbGYuaW5wdXRfdXJsKQogICAgICAgICAgICBpZiBwYXJzZWQudXNlcm5hbWUgb3IgcGFyc2VkLnBhc3N3b3JkIG9yIHBhcnNlZC5mcmFnbWVudDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVSTCBtdXN0IG5vdCBpbmNsdWRlIGNyZWRlbnRpYWxzIG9yIGEgZnJhZ21lbnQiKQogICAgICAgICAgICBmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgcGFyc2VfcXMKICAgICAgICAgICAgaWYgc2V0KHBhcnNlX3FzKHBhcnNlZC5xdWVyeSkpIC0geyJtb2RlbFZlcnNpb25JZCIsICJmaWxlSWQiLCAidHlwZSIsICJmb3JtYXQiLCAic2l6ZSIsICJmcCJ9OgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVtb3ZlIFVSTCB0b2tlbnM7IHVzZSB0aGUgbmFtZWQgQ29sYWIgc2VjcmV0IGZvciBhdXRoZW50aWNhdGlvbiIpCiAgICAgICAgaWYgc2VsZi5vdXRwdXRfbmFtZSAhPSBQYXRoKHNlbGYub3V0cHV0X25hbWUpLm5hbWUgb3Igbm90IHJlLmZ1bGxtYXRjaChyIltBLVphLXowLTldW0EtWmEtejAtOV8uLV0qXC5ja3B0Iiwgc2VsZi5vdXRwdXRfbmFtZSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm91dHB1dF9uYW1lIG11c3QgYmUgYSBzYWZlIGZpbGVuYW1lIGVuZGluZyBpbiAuY2twdCIpCiAgICAgICAgaWYgbm90IDEgPD0gc2VsZi5idWlsZF9qb2JzIDw9IDQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJ1aWxkX2pvYnMgbXVzdCBiZSBiZXR3ZWVuIDEgYW5kIDQiKQogICAgICAgIGlmIG5vdCAwIDwgc2VsZi5zY2FsZV9mYWN0b3IgPD0gMTA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNjYWxlX2ZhY3RvciBtdXN0IGJlIGdyZWF0ZXIgdGhhbiAwIGFuZCBhdCBtb3N0IDEwIikKICAgICAgICBpZiBzZWxmLm1vZGVsX3ZlcnNpb24gYW5kIG5vdCByZS5mdWxsbWF0Y2gociJbQS1aYS16MC05Xy4tXSsiLCBzZWxmLm1vZGVsX3ZlcnNpb24pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbF92ZXJzaW9uIG11c3QgYmUgYW4gdXBzdHJlYW0gRHJhdyBUaGluZ3MgdmVyc2lvbiBpZGVudGlmaWVyIikKICAgICAgICBpZiBub3Qgc2VsZi5ydW50aW1lX3Jvb3Quc3RyaXAoKSBvciBub3Qgc2VsZi5wZXJzaXN0ZW50X3Jvb3Quc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicnVudGltZV9yb290IGFuZCBwZXJzaXN0ZW50X3Jvb3QgbXVzdCBub3QgYmUgYmxhbmsiKQogICAgICAgIHBlcnNpc3RlbnQgPSBQYXRoKHNlbGYucGVyc2lzdGVudF9yb290KS5yZXNvbHZlKCkKICAgICAgICBydW50aW1lID0gUGF0aChzZWxmLnJ1bnRpbWVfcm9vdCkucmVzb2x2ZSgpCiAgICAgICAgcm9vdHNfb3ZlcmxhcCA9IEZhbHNlCiAgICAgICAgZm9yIGNoaWxkLCBwYXJlbnQgaW4gKChwZXJzaXN0ZW50LCBydW50aW1lKSwgKHJ1bnRpbWUsIHBlcnNpc3RlbnQpKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY2hpbGQucmVsYXRpdmVfdG8ocGFyZW50KQogICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJvb3RzX292ZXJsYXAgPSBUcnVlCiAgICAgICAgaWYgcm9vdHNfb3ZlcmxhcDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGVyc2lzdGVudF9yb290IGFuZCBydW50aW1lX3Jvb3QgbXVzdCBub3Qgb3ZlcmxhcCIpCgogICAgICAgIHJlcG9zaXRvcnkgPSB1cmxzcGxpdChzZWxmLnByb2plY3RfcmVwb3NpdG9yeV91cmwpIGlmIHNlbGYucHJvamVjdF9yZXBvc2l0b3J5X3VybCBlbHNlIE5vbmUKICAgICAgICBpZiByZXBvc2l0b3J5IGFuZCAocmVwb3NpdG9yeS51c2VybmFtZSBvciByZXBvc2l0b3J5LnBhc3N3b3JkIG9yIHJlcG9zaXRvcnkucXVlcnkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgInByb2plY3RfcmVwb3NpdG9yeV91cmwgbXVzdCBub3QgY29udGFpbiBjcmVkZW50aWFscyBvciBzaWduZWQgcXVlcnkgc3RyaW5ncyIKICAgICAgICAgICAgKQogICAgICAgIGlmIHNlbGYucHJvamVjdF9zb3VyY2VfbW9kZSA9PSAicmVwb3NpdG9yeSI6CiAgICAgICAgICAgIGlmIG5vdCBzZWxmLnByb2plY3RfcmVwb3NpdG9yeV91cmw6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJSZXBvc2l0b3J5IG1vZGUgcmVxdWlyZXMgcHJvamVjdF9yZXBvc2l0b3J5X3VybCIpCiAgICAgICAgICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbMC05YS1mQS1GXXs0MH0iLCBzZWxmLnByb2plY3RfcmV2aXNpb24pOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVwb3NpdG9yeSBtb2RlIHJlcXVpcmVzIGFuIGltbXV0YWJsZSA0MC1jaGFyYWN0ZXIgY29tbWl0IFNIQSIpCiAgICAgICAgaWYgc2VsZi5wcm9qZWN0X3NvdXJjZV9tb2RlID09ICJsb2NhbCIgYW5kICgKICAgICAgICAgICAgbm90IHNlbGYucHJvamVjdF9zb3VyY2VfcGF0aCBvciBzZWxmLnByb2plY3RfcmV2aXNpb24gIT0gImxvY2FsIgogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkxvY2FsIG1vZGUgcmVxdWlyZXMgcHJvamVjdF9zb3VyY2VfcGF0aCBhbmQgcHJvamVjdF9yZXZpc2lvbj0nbG9jYWwnIikKICAgICAgICBzdWJkaXJlY3RvcnkgPSBQdXJlUG9zaXhQYXRoKHNlbGYucHJvamVjdF9zdWJkaXJlY3Rvcnkgb3IgIi4iKQogICAgICAgIGlmIHN1YmRpcmVjdG9yeS5pc19hYnNvbHV0ZSgpIG9yICIuLiIgaW4gc3ViZGlyZWN0b3J5LnBhcnRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9qZWN0X3N1YmRpcmVjdG9yeSBtdXN0IGJlIGEgc2FmZSByZWxhdGl2ZSBwYXRoIikKCgpkZWYgdXRjX25vdygpIC0+IHN0cjoKICAgIHJldHVybiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKQoKCmRlZiBhdG9taWNfd3JpdGVfanNvbihwYXRoOiBQYXRoLCBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRlbXBvcmFyeSA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpCiAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgc3VibWl0dGVkX2pvYihjb25maWc6IEpvYkNvbmZpZywgKiwgc3VibWl0dGVkX3V0Yzogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgY29uZmlnLnZhbGlkYXRlKCkKICAgIHJldHVybiB7CiAgICAgICAgInBvbGljeV9yZXZpc2lvbiI6IFBPTElDWV9SRVZJU0lPTiwKICAgICAgICAic3VibWlzc2lvbl9pZCI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLnN0cmZ0aW1lKCIlWSVtJWRUJUglTSVTWiIpCiAgICAgICAgKyAiLSIKICAgICAgICArIHV1aWQudXVpZDQoKS5oZXhbOjhdLAogICAgICAgICJzdWJtaXR0ZWRfdXRjIjogc3VibWl0dGVkX3V0YyBvciB1dGNfbm93KCksCiAgICAgICAgInN1Ym1pdHRlZF9wcm9qZWN0X3JldmlzaW9uIjogKAogICAgICAgICAgICBjb25maWcucHJvamVjdF9yZXZpc2lvbgogICAgICAgICAgICBpZiBjb25maWcucHJvamVjdF9zb3VyY2VfbW9kZSAhPSAiZW1iZWRkZWQiCiAgICAgICAgICAgIGVsc2UgImVtYmVkZGVkLXRlbXBsYXRlIgogICAgICAgICksCiAgICAgICAgImNvbmZpZ3VyYXRpb24iOiBhc2RpY3QoY29uZmlnKSwKICAgIH0KCgpkZWYgc2F2ZV9wZW5kaW5nX2pvYihjb25maWc6IEpvYkNvbmZpZywgc3RhdGVfcm9vdDogUGF0aCkgLT4gUGF0aDoKICAgIHBlbmRpbmcgPSBzdGF0ZV9yb290IC8gInBlbmRpbmdfam9iLmpzb24iCiAgICBhdG9taWNfd3JpdGVfanNvbihwZW5kaW5nLCBzdWJtaXR0ZWRfam9iKGNvbmZpZykpCiAgICByZXR1cm4gcGVuZGluZwoKCmRlZiBsb2FkX3BlbmRpbmdfam9iKHBhdGg6IFBhdGgpIC0+IHR1cGxlW0pvYkNvbmZpZywgZGljdFtzdHIsIEFueV1dOgogICAgcGF5bG9hZCA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBpZiBwYXlsb2FkLmdldCgicG9saWN5X3JldmlzaW9uIikgIT0gUE9MSUNZX1JFVklTSU9OOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiUGVuZGluZyBqb2IgcG9saWN5IHtwYXlsb2FkLmdldCgncG9saWN5X3JldmlzaW9uJykhcn0gZG9lcyBub3QgbWF0Y2gge1BPTElDWV9SRVZJU0lPTiFyfSIKICAgICAgICApCiAgICBpZiAoCiAgICAgICAgbm90IHBheWxvYWQuZ2V0KCJzdWJtaXNzaW9uX2lkIikKICAgICAgICBvciBub3QgcGF5bG9hZC5nZXQoInN1Ym1pdHRlZF91dGMiKQogICAgICAgIG9yIG5vdCBwYXlsb2FkLmdldCgic3VibWl0dGVkX3Byb2plY3RfcmV2aXNpb24iKQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQZW5kaW5nIGpvYiBpcyBtaXNzaW5nIGltbXV0YWJsZSBzdWJtaXNzaW9uIGlkZW50aXR5IikKICAgIGNvbmZpZ3VyYXRpb24gPSBwYXlsb2FkLmdldCgiY29uZmlndXJhdGlvbiIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShjb25maWd1cmF0aW9uLCBkaWN0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQZW5kaW5nIGpvYiBoYXMgbm8gY29uZmlndXJhdGlvbiBvYmplY3QiKQogICAgY29uZmlnID0gSm9iQ29uZmlnKCoqY29uZmlndXJhdGlvbikKICAgIGNvbmZpZy52YWxpZGF0ZSgpCiAgICByZXR1cm4gY29uZmlnLCBwYXlsb2FkCg==", "src/project_app/converter.py": "IiIiQnVpbGQgYW5kIGludm9rZSB0aGUgb2ZmaWNpYWwgRHJhdyBUaGluZ3MgTG9SQUNvbnZlcnRlciBvbiBDUFUgTGludXguCgpUaGUgdXBzdHJlYW0gQ0xJIGFuZCBhcmd1bWVudHMgYXJlIHNvdXJjZS12ZXJpZmllZC4gVGhpcyBMaW51eC9Db2xhYiBidWlsZAppbnRlZ3JhdGlvbiBpcyBleHBlcmltZW50YWwgdW50aWwgYW4gYWN0dWFsIENvbGFiIGNvbnZlcnNpb24gc3VjY2VlZHMuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc2h1dGlsCmltcG9ydCBzcWxpdGUzCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0YXJmaWxlCmltcG9ydCB0ZW1wZmlsZQpmcm9tIHVybGxpYi5yZXF1ZXN0IGltcG9ydCB1cmxvcGVuCgpmcm9tIC5jYWNoZSBpbXBvcnQgY29udGVudF9maW5nZXJwcmludApmcm9tIC5jb25maWcgaW1wb3J0IGF0b21pY193cml0ZV9qc29uCmZyb20gLmxvZ2dpbmdfdXRpbHMgaW1wb3J0IHJ1bl9sb2dnZWRfY29tbWFuZAoKVVBTVFJFQU1fUkVWSVNJT04gPSAiOGQ5YzYyYWQzODUzOWVmOGJiMTIzZjhiOGU3OGNhYjc2Y2FhZmQwYyIKU1dJRlRfVkVSU0lPTiA9ICI2LjAuMyIKQkFaRUxfVkVSU0lPTiA9ICI3LjQuMSIKQlVJTERfUkVWSVNJT04gPSAiY3B1LWxpbnV4LXYyIgpTT1VSQ0VfUEFUQ0hfUkVWSVNJT04gPSAiemlwZm91bmRhdGlvbi1vcHRpb25hbC1maWxlLXYxIgpaSVBGT1VOREFUSU9OX09SSUdJTkFMX1NIQTI1NiA9ICczYmI2NDQ3ZGZmOWRkNTJiZDhkYTczMTExYzU5MmYwYTIxNzFmMDY3NTA4NjVkYzc5NWFhNTM0NTViZWIzYzAxJwpaSVBGT1VOREFUSU9OX1BBVENIRURfU0hBMjU2ID0gJ2ZjMDk1Njc0YmYzODVkYmZlM2QyYWUzMTYyM2E5YWNiMTgzZWRiMDU2MTZhM2U5ZmEyM2ViMTNmNDQ4MTJjNTAnCgoKZGVmIF9kb3dubG9hZCh1cmwsIHRhcmdldCk6CiAgICB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhcnRpYWwgPSB0YXJnZXQud2l0aF9zdWZmaXgodGFyZ2V0LnN1ZmZpeCArICIucGFydCIpCiAgICB3aXRoIHVybG9wZW4odXJsLCB0aW1lb3V0PTkwKSBhcyBzb3VyY2UsIHBhcnRpYWwub3Blbigid2IiKSBhcyBvdXRwdXQ6CiAgICAgICAgc2h1dGlsLmNvcHlmaWxlb2JqKHNvdXJjZSwgb3V0cHV0LCBsZW5ndGg9NCAqIDEwMjQgKiAxMDI0KQogICAgcGFydGlhbC5yZXBsYWNlKHRhcmdldCkKCgoKZGVmIHN3aWZ0X2Rpc3RyaWJ1dGlvbih1YnVudHUpOgogICAgIiIiUmV0dXJuIHRoZSBwaW5uZWQgVVJMIGFuZCBleHRyYWN0ZWQgZm9sZGVyIGZvciBhIHN1cHBvcnRlZCBVYnVudHUgcmVsZWFzZS4iIiIKICAgIGlmIHVidW50dSBub3QgaW4geyIyMi4wNCIsICIyNC4wNCJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkF1dG9tYXRpYyBTd2lmdCBzZXR1cCBzdXBwb3J0cyBVYnVudHUgMjIuMDQgYW5kIDI0LjA0IG9ubHkiKQogICAgZGlyZWN0b3J5ID0gInVidW50dSIgKyB1YnVudHUucmVwbGFjZSgiLiIsICIiKQogICAgZm9sZGVyID0gZiJzd2lmdC17U1dJRlRfVkVSU0lPTn0tUkVMRUFTRS11YnVudHV7dWJ1bnR1fSIKICAgIHVybCA9ICgKICAgICAgICBmImh0dHBzOi8vZG93bmxvYWQuc3dpZnQub3JnL3N3aWZ0LXtTV0lGVF9WRVJTSU9OfS1yZWxlYXNlLyIKICAgICAgICBmIntkaXJlY3Rvcnl9L3N3aWZ0LXtTV0lGVF9WRVJTSU9OfS1SRUxFQVNFL3tmb2xkZXJ9LnRhci5neiIKICAgICkKICAgIHJldHVybiB1cmwsIGZvbGRlcgoKCgpkZWYgcGF0Y2hfemlwZm91bmRhdGlvbihjaGVja291dCwgbG9nZ2VyKToKICAgICIiIlVud3JhcCB0aGUgb3B0aW9uYWwgRklMRSBwb2ludGVyIGluIHRoZSBwaW5uZWQgdmVuZG9yZWQgWklQRm91bmRhdGlvbiBzb3VyY2UuIiIiCiAgICBzb3VyY2UgPSBjaGVja291dCAvICJWZW5kb3JzL1pJUEZvdW5kYXRpb24vU291cmNlcy9aSVBGb3VuZGF0aW9uL0FyY2hpdmUrTWVtb3J5RmlsZS5zd2lmdCIKICAgIGRpZ2VzdCA9IGNvbnRlbnRfZmluZ2VycHJpbnQoc291cmNlKQogICAgaWYgZGlnZXN0ID09IFpJUEZPVU5EQVRJT05fUEFUQ0hFRF9TSEEyNTY6CiAgICAgICAgbG9nZ2VyLmluZm8oIlNPVVJDRSBQQVRDSCBSRVVTRUQ6ICVzIiwgU09VUkNFX1BBVENIX1JFVklTSU9OKQogICAgICAgIHJldHVybgogICAgaWYgZGlnZXN0ICE9IFpJUEZPVU5EQVRJT05fT1JJR0lOQUxfU0hBMjU2OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiWklQRm91bmRhdGlvbiBzb3VyY2UgZG9lcyBub3QgbWF0Y2ggdGhlIHBpbm5lZCBjb21wYXRpYmlsaXR5IHBhdGNoIikKICAgIG9yaWdpbmFsID0gc291cmNlLnJlYWRfdGV4dCgpCiAgICBiZWZvcmUgPSAiICAgICAgaWYgYXBwZW5kIHtcbiAgICAgICAgZnNlZWtvKHJlc3VsdCwgMCwgU0VFS19FTkQpIgogICAgYWZ0ZXIgPSAiICAgICAgaWYgYXBwZW5kLCBsZXQgcmVzdWx0ID0gcmVzdWx0IHtcbiAgICAgICAgZnNlZWtvKHJlc3VsdCwgMCwgU0VFS19FTkQpIgogICAgaWYgb3JpZ2luYWwuY291bnQoYmVmb3JlKSAhPSAxOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiWklQRm91bmRhdGlvbiBvcHRpb25hbCBGSUxFIHBhdGNoIG5vIGxvbmdlciBtYXRjaGVzIikKICAgIHNvdXJjZS53cml0ZV90ZXh0KG9yaWdpbmFsLnJlcGxhY2UoYmVmb3JlLCBhZnRlciwgMSkpCiAgICBpZiBjb250ZW50X2ZpbmdlcnByaW50KHNvdXJjZSkgIT0gWklQRk9VTkRBVElPTl9QQVRDSEVEX1NIQTI1NjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlpJUEZvdW5kYXRpb24gY29tcGF0aWJpbGl0eSBwYXRjaCBjaGVja3N1bSBtaXNtYXRjaCIpCiAgICBsb2dnZXIuaW5mbygiU09VUkNFIFBBVENIIEFQUExJRUQ6ICVzIiwgU09VUkNFX1BBVENIX1JFVklTSU9OKQoKCmRlZiBwcmVwYXJlX2NvbnZlcnRlcihyb290LCBqb2JzLCBsb2dnZXIpOgogICAgaWYgcGxhdGZvcm0uc3lzdGVtKCkgIT0gIkxpbnV4IiBvciBwbGF0Zm9ybS5tYWNoaW5lKCkgIT0gIng4Nl82NCI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgYXV0b21hdGljIGNvbnZlcnRlciBidWlsZCB0YXJnZXRzIHg4Nl82NCBMaW51eCBvbmx5IikKICAgIHJvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgYmluYXJ5ID0gcm9vdCAvICJMb1JBQ29udmVydGVyIgogICAgcmVjZWlwdCA9IHJvb3QgLyAiYnVpbGRfcmVjZWlwdC5qc29uIgogICAgaWRlbnRpdHkgPSB7InVwc3RyZWFtIjogVVBTVFJFQU1fUkVWSVNJT04sICJzd2lmdCI6IFNXSUZUX1ZFUlNJT04sICJiYXplbCI6IEJBWkVMX1ZFUlNJT04sICJidWlsZCI6IEJVSUxEX1JFVklTSU9OLCAicGxhdGZvcm0iOiBwbGF0Zm9ybS5wbGF0Zm9ybSgpLCAiY190b29sY2hhaW4iOiAic3lzdGVtLWNsYW5nLW9wZW5tcC12MSIsICJzb3VyY2VfcGF0Y2giOiBTT1VSQ0VfUEFUQ0hfUkVWSVNJT059CiAgICBpZiBiaW5hcnkuaXNfZmlsZSgpIGFuZCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBwcmV2aW91cyA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgICAgICBpZiBwcmV2aW91cy5nZXQoImlkZW50aXR5IikgPT0gaWRlbnRpdHkgYW5kIHByZXZpb3VzLmdldCgic2hhMjU2IikgPT0gY29udGVudF9maW5nZXJwcmludChiaW5hcnkpOgogICAgICAgICAgICBydW5fbG9nZ2VkX2NvbW1hbmQoW3N0cihiaW5hcnkpLCAiLS1oZWxwIl0sIGxvZ2dlciwgZW52PWNvbnZlcnRlcl9lbnYocm9vdCkpCiAgICAgICAgICAgIHJldHVybiBiaW5hcnkKICAgIGxvZ2dlci5pbmZvKCJCVUlMRCBTVEFSVDogb2ZmaWNpYWwgTG9SQUNvbnZlcnRlcjsgZXhwZXJpbWVudGFsIENQVSBMaW51eCBidWlsZDsgbm8gbW9kZWwgd2VpZ2h0cyBhcmUgbmVlZGVkIikKICAgIHJ1bl9sb2dnZWRfY29tbWFuZChbImFwdC1nZXQiLCAiaW5zdGFsbCIsICIteSIsICJnaXQiLCAiY2xhbmciLCAibGx2bSIsICJsaWJvbXAtZGV2IiwgImNtYWtlIiwgInBrZy1jb25maWciLCAibGlicG5nLWRldiIsICJsaWJqcGVnLWRldiIsICJsaWJvcGVuYmxhcy1kZXYiLCAibGliZWRpdDIiLCAibGlieG1sMiIsICJsaWJzcWxpdGUzLTAiLCAibGliY3VybDQtb3BlbnNzbC1kZXYiLCAibGlic3NsLWRldiIsICJ6bGliMWctZGV2IiwgImxpYm5jdXJzZXM2IiwgImxpYmF0b21pYzEiLCAibGlicHl0aG9uMy1kZXYiXSwgbG9nZ2VyKQogICAgIyBVc2UgVWJ1bnR1J3MgY29tcGlsZXIgYW5kIGl0cyBtYXRjaGluZyBPcGVuTVAgZGV2ZWxvcG1lbnQgcGFja2FnZS4KICAgICMgU3dpZnQncyBiaW4gZGlyZWN0b3J5IGFsc28gY29udGFpbnMgY2xhbmcsIGJ1dCBub3QgVWJ1bnR1J3Mgb21wLmguCiAgICBjYyA9IFBhdGgoIi91c3IvYmluL2NsYW5nIikKICAgIGN4eCA9IFBhdGgoIi91c3IvYmluL2NsYW5nKysiKQogICAgaWYgbm90IGNjLmlzX2ZpbGUoKSBvciBub3QgY3h4LmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlN5c3RlbSBDbGFuZyB3YXMgbm90IGluc3RhbGxlZCBzdWNjZXNzZnVsbHkiKQogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkocHJlZml4PSJvcGVubXAtY2hlY2stIiwgZGlyPXJvb3QpIGFzIGNoZWNrOgogICAgICAgIHNvdXJjZSA9IFBhdGgoY2hlY2spIC8gIm9wZW5tcC5jIgogICAgICAgIGV4ZWN1dGFibGUgPSBQYXRoKGNoZWNrKSAvICJvcGVubXAtY2hlY2siCiAgICAgICAgc291cmNlLndyaXRlX3RleHQoIiNpbmNsdWRlIDxvbXAuaD5cbmludCBtYWluKHZvaWQpIHsgcmV0dXJuIG9tcF9nZXRfbWF4X3RocmVhZHMoKSA8IDE7IH1cbiIpCiAgICAgICAgbG9nZ2VyLmluZm8oIlRPT0xDSEFJTiBDSEVDSzogc3lzdGVtIENsYW5nIE9wZW5NUCBjb21waWxlLCBsaW5rIGFuZCBleGVjdXRpb24iKQogICAgICAgIHJ1bl9sb2dnZWRfY29tbWFuZChbc3RyKGNjKSwgIi1mb3Blbm1wIiwgc3RyKHNvdXJjZSksICItbyIsIHN0cihleGVjdXRhYmxlKV0sIGxvZ2dlcikKICAgICAgICBydW5fbG9nZ2VkX2NvbW1hbmQoW3N0cihleGVjdXRhYmxlKV0sIGxvZ2dlcikKICAgIHJlbGVhc2UgPSBwbGF0Zm9ybS5mcmVlZGVza3RvcF9vc19yZWxlYXNlKCkKICAgIHVidW50dSA9IHJlbGVhc2UuZ2V0KCJWRVJTSU9OX0lEIikKICAgIGlmIHJlbGVhc2UuZ2V0KCJJRCIpICE9ICJ1YnVudHUiIG9yIHVidW50dSBub3QgaW4geyIyMi4wNCIsICIyNC4wNCJ9OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQXV0b21hdGljIFN3aWZ0IHNldHVwIHN1cHBvcnRzIFVidW50dSAyMi4wNCBhbmQgMjQuMDQgb25seSIpCiAgICBzd2lmdF91cmwsIHN3aWZ0X2ZvbGRlciA9IHN3aWZ0X2Rpc3RyaWJ1dGlvbih1YnVudHUpCiAgICB0b29scyA9IHJvb3QgLyAidG9vbGNoYWluIgogICAgc3dpZnRfcm9vdCA9IHRvb2xzIC8gc3dpZnRfZm9sZGVyCiAgICBpZiBub3QgKHN3aWZ0X3Jvb3QgLyAidXNyL2Jpbi9zd2lmdGMiKS5pc19maWxlKCk6CiAgICAgICAgYXJjaGl2ZSA9IHRvb2xzIC8gInN3aWZ0LnRhci5neiIKICAgICAgICBsb2dnZXIuaW5mbygiVE9PTENIQUlOIERPV05MT0FEOiBTd2lmdCAlcyBmb3IgVWJ1bnR1ICVzIiwgU1dJRlRfVkVSU0lPTiwgdWJ1bnR1KQogICAgICAgIF9kb3dubG9hZChzd2lmdF91cmwsIGFyY2hpdmUpCiAgICAgICAgd2l0aCB0YXJmaWxlLm9wZW4oYXJjaGl2ZSkgYXMgdGFyOgogICAgICAgICAgICB0YXIuZXh0cmFjdGFsbCh0b29scywgZmlsdGVyPSJkYXRhIikKICAgICAgICBpZiBub3QgKHN3aWZ0X3Jvb3QgLyAidXNyL2Jpbi9zd2lmdGMiKS5pc19maWxlKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlN3aWZ0IGFyY2hpdmUgZGlkIG5vdCBjb250YWluIHtzd2lmdF9mb2xkZXJ9L3Vzci9iaW4vc3dpZnRjIikKICAgIGJhemVsID0gdG9vbHMgLyAiYmF6ZWwiCiAgICBpZiBub3QgYmF6ZWwuaXNfZmlsZSgpOgogICAgICAgIF9kb3dubG9hZChmImh0dHBzOi8vZ2l0aHViLmNvbS9iYXplbGJ1aWxkL2JhemVsL3JlbGVhc2VzL2Rvd25sb2FkL3tCQVpFTF9WRVJTSU9OfS9iYXplbC17QkFaRUxfVkVSU0lPTn0tbGludXgteDg2XzY0IiwgYmF6ZWwpCiAgICAgICAgYmF6ZWwuY2htb2QoMG83NTUpCiAgICBjaGVja291dCA9IHJvb3QgLyAic291cmNlIgogICAgaWYgbm90IChjaGVja291dCAvICIuZ2l0IikuaXNfZGlyKCk6CiAgICAgICAgcnVuX2xvZ2dlZF9jb21tYW5kKFsiZ2l0IiwgImluaXQiLCBzdHIoY2hlY2tvdXQpXSwgbG9nZ2VyKQogICAgICAgIHJ1bl9sb2dnZWRfY29tbWFuZChbImdpdCIsICItQyIsIHN0cihjaGVja291dCksICJyZW1vdGUiLCAiYWRkIiwgIm9yaWdpbiIsICJodHRwczovL2dpdGh1Yi5jb20vZHJhd3RoaW5nc2FpL2RyYXctdGhpbmdzLWNvbW11bml0eS5naXQiXSwgbG9nZ2VyKQogICAgcnVuX2xvZ2dlZF9jb21tYW5kKFsiZ2l0IiwgIi1DIiwgc3RyKGNoZWNrb3V0KSwgImZldGNoIiwgIi0tZGVwdGgiLCAiMSIsICJvcmlnaW4iLCBVUFNUUkVBTV9SRVZJU0lPTl0sIGxvZ2dlcikKICAgIHJ1bl9sb2dnZWRfY29tbWFuZChbImdpdCIsICItQyIsIHN0cihjaGVja291dCksICJjaGVja291dCIsICItLWRldGFjaCIsIFVQU1RSRUFNX1JFVklTSU9OXSwgbG9nZ2VyKQogICAgYWN0dWFsID0gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoWyJnaXQiLCAiLUMiLCBzdHIoY2hlY2tvdXQpLCAicmV2LXBhcnNlIiwgIkhFQUQiXSwgdGV4dD1UcnVlKS5zdHJpcCgpCiAgICBpZiBhY3R1YWwgIT0gVVBTVFJFQU1fUkVWSVNJT046CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJVcHN0cmVhbSBzb3VyY2UgcmV2aXNpb24gbWlzbWF0Y2giKQogICAgcGF0Y2hfemlwZm91bmRhdGlvbihjaGVja291dCwgbG9nZ2VyKQogICAgIyBUaGUgb2ZmaWNpYWwgTGludXggd29ya3NwYWNlIGRlZmF1bHRzIHRvIGN1RE5OLiBDb252ZXJzaW9uIHVzZXMgQ1BVOwogICAgIyBjaGFuZ2Ugb25seSB0aGlzIGdlbmVyYXRlZCB3b3Jrc3BhY2UgY29weSBhbmQgcHJlc2VydmUgcHJvdmVuYW5jZS4KICAgIHdvcmtzcGFjZSA9IChjaGVja291dCAvICJXT1JLU1BBQ0UubGludXgiKS5yZWFkX3RleHQoKQogICAgaWYgd29ya3NwYWNlLmNvdW50KCJoYXZlX2N1ZG5uID0gVHJ1ZSIpICE9IDE6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJVcHN0cmVhbSBDUFUgYnVpbGQgYWRhcHRhdGlvbiBubyBsb25nZXIgbWF0Y2hlcyIpCiAgICAoY2hlY2tvdXQgLyAiV09SS1NQQUNFIikud3JpdGVfdGV4dCh3b3Jrc3BhY2UucmVwbGFjZSgiaGF2ZV9jdWRubiA9IFRydWUiLCAiaGF2ZV9jdWRubiA9IEZhbHNlIikpCiAgICAoY2hlY2tvdXQgLyAiLmJhemVscmMiKS53cml0ZV90ZXh0KCJ0cnktaW1wb3J0ICV3b3Jrc3BhY2UlLy5iYXplbHJjLmxpbnV4XG4iKQogICAgZW52ID0gZGljdChvcy5lbnZpcm9uLCBQQVRIPXN0cihzd2lmdF9yb290IC8gInVzci9iaW4iKSArIG9zLnBhdGhzZXAgKyBvcy5lbnZpcm9uLmdldCgiUEFUSCIsICIiKSwgQ0M9c3RyKGNjKSwgQ1hYPXN0cihjeHgpKQogICAgcnVuX2xvZ2dlZF9jb21tYW5kKFtzdHIoYmF6ZWwpLCAiYnVpbGQiLCAiQXBwczpMb1JBQ29udmVydGVyIiwgIi0tY29tcGlsYXRpb25fbW9kZT1vcHQiLCBmIi0tam9icz17am9ic30iLCAiLS1zcGF3bl9zdHJhdGVneT1sb2NhbCIsIGYiLS1yZXBvX2Vudj1DQz17Y2N9IiwgZiItLXJlcG9fZW52PUNYWD17Y3h4fSIsICItLXZlcmJvc2VfZmFpbHVyZXMiXSwgbG9nZ2VyLCBjd2Q9Y2hlY2tvdXQsIGVudj1lbnYpCiAgICBidWlsdCA9IGNoZWNrb3V0IC8gImJhemVsLWJpbi9BcHBzL0xvUkFDb252ZXJ0ZXIiCiAgICBydW5fbG9nZ2VkX2NvbW1hbmQoW3N0cihidWlsdCksICItLWhlbHAiXSwgbG9nZ2VyLCBlbnY9ZW52KQogICAgc2h1dGlsLmNvcHkyKGJ1aWx0LCBiaW5hcnkpCiAgICAjIENvcHkgdGhlIExpbnV4IHJ1bnRpbWUgbGlicmFyaWVzIG5lZWRlZCBieSBTd2lmdCdzIGR5bmFtaWNhbGx5LWxpbmtlZCBiaW5hcnkuCiAgICBsaWJyYXJpZXMgPSBzd2lmdF9yb290IC8gInVzci9saWIvc3dpZnQvbGludXgiCiAgICB0YXJnZXRfbGliID0gcm9vdCAvICJzd2lmdC1saWJzIgogICAgc2h1dGlsLmNvcHl0cmVlKGxpYnJhcmllcywgdGFyZ2V0X2xpYiwgZGlyc19leGlzdF9vaz1UcnVlKQogICAgcnVuX2xvZ2dlZF9jb21tYW5kKFtzdHIoYmluYXJ5KSwgIi0taGVscCJdLCBsb2dnZXIsIGVudj1jb252ZXJ0ZXJfZW52KHJvb3QpKQogICAgYXRvbWljX3dyaXRlX2pzb24ocmVjZWlwdCwgeyJpZGVudGl0eSI6IGlkZW50aXR5LCAic2hhMjU2IjogY29udGVudF9maW5nZXJwcmludChiaW5hcnkpLCAic291cmNlX3BhdGNoIjogU09VUkNFX1BBVENIX1JFVklTSU9OLCAiYWRhcHRhdGlvbiI6ICJXT1JLU1BBQ0U6IGhhdmVfY3Vkbm49RmFsc2U7IHN5c3RlbSBDbGFuZyB3aXRoIGxpYm9tcC1kZXY7IFpJUEZvdW5kYXRpb24gb3B0aW9uYWwgRklMRSB1bndyYXA7IG5vIENVREEgYnVpbGQgY29uZmlnIn0pCiAgICByZXR1cm4gYmluYXJ5CgoKZGVmIGNvbnZlcnRlcl9lbnYocm9vdCk6CiAgICByZXR1cm4gZGljdChvcy5lbnZpcm9uLCBMRF9MSUJSQVJZX1BBVEg9c3RyKHJvb3QgLyAic3dpZnQtbGlicyIpICsgb3MucGF0aHNlcCArIG9zLmVudmlyb24uZ2V0KCJMRF9MSUJSQVJZX1BBVEgiLCAiIikpCgoKZGVmIHZhbGlkYXRlX2NoZWNrcG9pbnQocGF0aCk6CiAgICAiIiJSZWplY3QgcGlja2xlL3JlbmFtZWQgc2FmZXRlbnNvcnM7IHZlcmlmeSB0aGUgRHJhdyBUaGluZ3MgU1FMaXRlIHN0b3JlLiIiIgogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgIGlmIHN0cmVhbS5yZWFkKDE2KSAhPSBiIlNRTGl0ZSBmb3JtYXQgM1x4MDAiOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDb252ZXJ0ZXIgb3V0cHV0IGlzIG5vdCBhIERyYXcgVGhpbmdzIFNRTGl0ZSBjaGVja3BvaW50IikKICAgIHdpdGggc3FsaXRlMy5jb25uZWN0KHBhdGgucmVzb2x2ZSgpLmFzX3VyaSgpICsgIj9tb2RlPXJvIiwgdXJpPVRydWUpIGFzIGRiOgogICAgICAgIGlmIGRiLmV4ZWN1dGUoIlBSQUdNQSBxdWlja19jaGVjayIpLmZldGNob25lKCkgIT0gKCJvayIsKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29udmVydGVkIGNoZWNrcG9pbnQgZmFpbGVkIFNRTGl0ZSBpbnRlZ3JpdHkgdmFsaWRhdGlvbiIpCiAgICAgICAgaWYgbm90IGRiLmV4ZWN1dGUoIlNFTEVDVCBuYW1lIEZST00gc3FsaXRlX21hc3RlciBXSEVSRSB0eXBlPSd0YWJsZSciKS5mZXRjaGFsbCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDb252ZXJ0ZWQgY2hlY2twb2ludCBjb250YWlucyBubyB0YWJsZXMiKQogICAgaWYgcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgIi10ZW5zb3JkYXRhIikuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5leHBlY3RlZCBleHRlcm5hbCB0ZW5zb3Igc2lkZWNhcjsgdGhpcyB3b3JrZmxvdyBvbmx5IGRlbGl2ZXJzIHNlbGYtY29udGFpbmVkIExvUkEgY2hlY2twb2ludHMiKQoKCmRlZiBjb252ZXJ0KGJpbmFyeSwgc291cmNlLCB0YXJnZXQsIGNvbmZpZywgbG9nZ2VyKToKICAgIHRhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkocHJlZml4PSJsb3JhLWNvbnZlcnQtIiwgZGlyPVBhdGgoY29uZmlnLnJ1bnRpbWVfcm9vdCkpIGFzIHRlbXBvcmFyeToKICAgICAgICBvdXRwdXQgPSBQYXRoKHRlbXBvcmFyeSkKICAgICAgICBjb21tYW5kID0gW3N0cihiaW5hcnkpLCAiLS1maWxlIiwgc3RyKHNvdXJjZSksICItLW5hbWUiLCAiY29udmVydGVkIiwgIi0tb3V0cHV0LWRpcmVjdG9yeSIsIHN0cihvdXRwdXQpLCAiLS1zY2FsZS1mYWN0b3IiLCBzdHIoY29uZmlnLnNjYWxlX2ZhY3RvcildCiAgICAgICAgaWYgY29uZmlnLm1vZGVsX3ZlcnNpb246CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLXZlcnNpb24iLCBjb25maWcubW9kZWxfdmVyc2lvbl0KICAgICAgICBydW5fbG9nZ2VkX2NvbW1hbmQoY29tbWFuZCwgbG9nZ2VyLCBlbnY9Y29udmVydGVyX2VudihiaW5hcnkucGFyZW50KSkKICAgICAgICBjaGVja3BvaW50ID0gb3V0cHV0IC8gImNvbnZlcnRlZF9sb3JhX2YxNi5ja3B0IgogICAgICAgIGlmIG5vdCBjaGVja3BvaW50LmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJPZmZpY2lhbCBjb252ZXJ0ZXIgZGlkIG5vdCBjcmVhdGUgdGhlIGV4cGVjdGVkIGNoZWNrcG9pbnQiKQogICAgICAgIHZhbGlkYXRlX2NoZWNrcG9pbnQoY2hlY2twb2ludCkKICAgICAgICBzaHV0aWwuY29weTIoY2hlY2twb2ludCwgdGFyZ2V0KQogICAgICAgIHZhbGlkYXRlX2NoZWNrcG9pbnQodGFyZ2V0KQo=", "src/project_app/downloads.py": "IiIiQ2l2aXRhaSB2ZXJzaW9uIHJlc29sdXRpb24gYW5kIGNoZWNrc3VtLXZlcmlmaWVkLCByZXN1bWFibGUgTG9SQSBkb3dubG9hZHMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHN0cnVjdApmcm9tIHVybGxpYi5lcnJvciBpbXBvcnQgSFRUUEVycm9yLCBVUkxFcnJvcgpmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgcGFyc2VfcXMsIHVybHNwbGl0CmZyb20gdXJsbGliLnJlcXVlc3QgaW1wb3J0IEhUVFBSZWRpcmVjdEhhbmRsZXIsIFJlcXVlc3QsIGJ1aWxkX29wZW5lcgoKZnJvbSAuY2FjaGUgaW1wb3J0IGNvbnRlbnRfZmluZ2VycHJpbnQKZnJvbSAuY29uZmlnIGltcG9ydCBDSVZJVEFJX0hPU1RTLCBjaXZpdGFpX29yaWdpbgoKCmNsYXNzIFNhZmVSZWRpcmVjdChIVFRQUmVkaXJlY3RIYW5kbGVyKToKICAgIGRlZiByZWRpcmVjdF9yZXF1ZXN0KHNlbGYsIHJlcSwgZnAsIGNvZGUsIG1zZywgaGVhZGVycywgbmV3dXJsKToKICAgICAgICBwYXJzZWQgPSB1cmxzcGxpdChuZXd1cmwpCiAgICAgICAgaWYgcGFyc2VkLnNjaGVtZSAhPSAiaHR0cHMiIG9yIHBhcnNlZC51c2VybmFtZSBvciBwYXJzZWQucGFzc3dvcmQ6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUmVmdXNpbmcgYW4gdW5zYWZlIGRvd25sb2FkIHJlZGlyZWN0IikKICAgICAgICByZXN1bHQgPSBzdXBlcigpLnJlZGlyZWN0X3JlcXVlc3QocmVxLCBmcCwgY29kZSwgbXNnLCBoZWFkZXJzLCBuZXd1cmwpCiAgICAgICAgaWYgcmVzdWx0IGlzIG5vdCBOb25lIGFuZCBwYXJzZWQuaG9zdG5hbWUgIT0gdXJsc3BsaXQocmVxLmZ1bGxfdXJsKS5ob3N0bmFtZToKICAgICAgICAgICAgcmVzdWx0LnJlbW92ZV9oZWFkZXIoIkF1dGhvcml6YXRpb24iKQogICAgICAgIHJldHVybiByZXN1bHQKCgpkZWYgb3Blbl91cmwodXJsLCB0b2tlbj0iIiwgKiwgaGVhZGVycz1Ob25lKToKICAgICIiIkF1dGhvcml6ZSBvbmx5IENpdml0YWk7IG5ldmVyIGZvcndhcmQgY3JlZGVudGlhbHMgdG8gaXRzIGRvd25sb2FkIENETi4iIiIKICAgIHBhcnNlZCA9IHVybHNwbGl0KHVybCkKICAgIGlmIHBhcnNlZC5zY2hlbWUgIT0gImh0dHBzIjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJIVFRQUyBpcyByZXF1aXJlZCIpCiAgICB2YWx1ZXMgPSB7IlVzZXItQWdlbnQiOiAiRHJhd1RoaW5ncy1Mb1JBLUNvbGFiLzEuMCIsICoqKGhlYWRlcnMgb3Ige30pfQogICAgaWYgdG9rZW4gYW5kIHBhcnNlZC5ob3N0bmFtZSBpbiBDSVZJVEFJX0hPU1RTOgogICAgICAgIHZhbHVlc1siQXV0aG9yaXphdGlvbiJdID0gIkJlYXJlciAiICsgdG9rZW4KICAgIHRyeToKICAgICAgICByZXR1cm4gYnVpbGRfb3BlbmVyKFNhZmVSZWRpcmVjdCgpKS5vcGVuKFJlcXVlc3QodXJsLCBoZWFkZXJzPXZhbHVlcyksIHRpbWVvdXQ9OTApCiAgICBleGNlcHQgSFRUUEVycm9yIGFzIGV4YzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDaXZpdGFpL2Rvd25sb2FkIEhUVFAge2V4Yy5jb2RlfTsgY2hlY2sgdGhlIFVSTCwgYWNjZXNzIHJpZ2h0cyBhbmQgQ29sYWIgc2VjcmV0IikgZnJvbSBOb25lCiAgICBleGNlcHQgVVJMRXJyb3I6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOZXR3b3JrIHJlcXVlc3QgZmFpbGVkOyByZXRyeSB0aGUgc2FtZSBzdWJtaXR0ZWQgam9iIikgZnJvbSBOb25lCgoKZGVmIGFwaV9qc29uKHBhdGgsIHRva2VuLCAqLCBvcmlnaW49Imh0dHBzOi8vY2l2aXRhaS5jb20iKToKICAgIHdpdGggb3Blbl91cmwob3JpZ2luICsgIi9hcGkvdjEvIiArIHBhdGgsIHRva2VuKSBhcyByZXNwb25zZToKICAgICAgICBkYXRhID0gcmVzcG9uc2UucmVhZCg4XzAwMF8wMDEpCiAgICAgICAgaWYgbGVuKGRhdGEpID4gOF8wMDBfMDAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaXZpdGFpIG1ldGFkYXRhIHJlc3BvbnNlIGlzIHVuZXhwZWN0ZWRseSBsYXJnZSIpCiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMoZGF0YSkKCgpkZWYgcmVzb2x2ZV9sb3JhKHVybCwgdG9rZW49IiIpOgogICAgb3JpZ2luID0gY2l2aXRhaV9vcmlnaW4odXJsKQogICAgcGFyc2VkID0gdXJsc3BsaXQodXJsLnN0cmlwKCkpCiAgICBxdWVyeSA9IHBhcnNlX3FzKHBhcnNlZC5xdWVyeSkKICAgIG1hdGNoID0gcmUuZnVsbG1hdGNoKHIiL2FwaS9kb3dubG9hZC9tb2RlbHMvKFxkKykvPyIsIHBhcnNlZC5wYXRoKQogICAgZmlsZV9pZHMgPSBxdWVyeS5nZXQoImZpbGVJZCIsIFtdKQogICAgaWYgZmlsZV9pZHMgYW5kIChsZW4oZmlsZV9pZHMpICE9IDEgb3Igbm90IGZpbGVfaWRzWzBdLmlzZGlnaXQoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmlsZUlkIG11c3QgYmUgb25lIG51bWVyaWMgQ2l2aXRhaSBmaWxlIElEIikKICAgIGZpbGVfaWQgPSBmaWxlX2lkc1swXSBpZiBmaWxlX2lkcyBlbHNlIE5vbmUKICAgIGlmIG1hdGNoOgogICAgICAgIHZlcnNpb25faWQgPSBtYXRjaFsxXQogICAgZWxzZToKICAgICAgICBtYXRjaCA9IHJlLm1hdGNoKHIiXi9tb2RlbHMvKFxkKykoPzovW14vXSopPy8/JCIsIHBhcnNlZC5wYXRoKQogICAgICAgIGlmIG5vdCBtYXRjaDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVXNlIGEgQ2l2aXRhaSBtb2RlbCBwYWdlIG9yIC9hcGkvZG93bmxvYWQvbW9kZWxzL1ZFUlNJT05fSUQgVVJMIikKICAgICAgICBpZiBxdWVyeS5nZXQoIm1vZGVsVmVyc2lvbklkIik6CiAgICAgICAgICAgIHZlcnNpb25faWQgPSBxdWVyeVsibW9kZWxWZXJzaW9uSWQiXVswXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG1vZGVsID0gYXBpX2pzb24oIm1vZGVscy8iICsgbWF0Y2hbMV0sIHRva2VuLCBvcmlnaW49b3JpZ2luKQogICAgICAgICAgICB2ZXJzaW9ucyA9IG1vZGVsLmdldCgibW9kZWxWZXJzaW9ucyIsIFtdKQogICAgICAgICAgICBpZiBub3QgdmVyc2lvbnM6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDaXZpdGFpIG1vZGVsIGhhcyBubyBhY2Nlc3NpYmxlIHZlcnNpb25zIikKICAgICAgICAgICAgdmVyc2lvbl9pZCA9IHN0cih2ZXJzaW9uc1swXVsiaWQiXSkKICAgIGlmIG5vdCBzdHIodmVyc2lvbl9pZCkuaXNkaWdpdCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgbW9kZWwgdmVyc2lvbiBJRCIpCiAgICB2ZXJzaW9uID0gYXBpX2pzb24oIm1vZGVsLXZlcnNpb25zLyIgKyBzdHIodmVyc2lvbl9pZCksIHRva2VuLCBvcmlnaW49b3JpZ2luKQogICAgaWYgdmVyc2lvbi5nZXQoIm1vZGVsIiwge30pLmdldCgidHlwZSIsICIiKS51cHBlcigpIG5vdCBpbiB7IkxPUkEiLCAiTE9DT04ifToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaGlzIHdvcmtmbG93IGFjY2VwdHMgTG9SQS9Mb0NvbiBhc3NldHMsIG5vdCBmdWxsIGNoZWNrcG9pbnRzIikKICAgIGZpbGVzID0gW2YgZm9yIGYgaW4gdmVyc2lvbi5nZXQoImZpbGVzIiwgW10pIGlmIGYuZ2V0KCJuYW1lIiwgIiIpLmxvd2VyKCkuZW5kc3dpdGgoIi5zYWZldGVuc29ycyIpIGFuZCBmLmdldCgidHlwZSIpID09ICJNb2RlbCJdCiAgICBmb3Iga2V5IGluICgic2l6ZSIsICJmcCIsICJmb3JtYXQiKToKICAgICAgICBpZiBxdWVyeS5nZXQoa2V5KToKICAgICAgICAgICAgZmlsZXMgPSBbZiBmb3IgZiBpbiBmaWxlcyBpZiBzdHIoZi5nZXQoIm1ldGFkYXRhIiwge30pLmdldChrZXksICIiKSkubG93ZXIoKSA9PSBxdWVyeVtrZXldWzBdLmxvd2VyKCldCiAgICBpZiBmaWxlX2lkIGlzIG5vdCBOb25lOgogICAgICAgIGZpbGVzID0gW2YgZm9yIGYgaW4gZmlsZXMgaWYgc3RyKGYuZ2V0KCJpZCIpKSA9PSBmaWxlX2lkXQogICAgcHJpbWFyeSA9IFtmIGZvciBmIGluIGZpbGVzIGlmIGYuZ2V0KCJwcmltYXJ5IildCiAgICBmaWxlcyA9IHByaW1hcnkgb3IgZmlsZXMKICAgIGlmIGxlbihmaWxlcykgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJObyB1bmlxdWUgc2FmZXRlbnNvcnMgZmlsZSB3YXMgZm91bmQ7IHVzZSBhIHZlcnNpb24tc3BlY2lmaWMgZG93bmxvYWQgVVJMIHdpdGggZmlsZSB2YXJpYW50IHBhcmFtZXRlcnMiKQogICAgc2VsZWN0ZWQgPSBmaWxlc1swXQogICAgZGlnZXN0ID0gc2VsZWN0ZWQuZ2V0KCJoYXNoZXMiLCB7fSkuZ2V0KCJTSEEyNTYiLCAiIikubG93ZXIoKQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyIlswLTlhLWZdezY0fSIsIGRpZ2VzdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQSBDaXZpdGFpIFNIQS0yNTYgaXMgcmVxdWlyZWQgdG8gdmVyaWZ5IGFuZCBzYWZlbHkgcmVzdW1lIHRoaXMgZG93bmxvYWQiKQogICAgZG93bmxvYWRfdXJsID0gc2VsZWN0ZWRbImRvd25sb2FkVXJsIl0KICAgIGlmIHVybHNwbGl0KGRvd25sb2FkX3VybCkuaG9zdG5hbWUgbm90IGluIENJVklUQUlfSE9TVFM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5leHBlY3RlZCBDaXZpdGFpIEFQSSBkb3dubG9hZCBVUkwiKQogICAgY2l2aXRhaV9vcmlnaW4oZG93bmxvYWRfdXJsKSAgIyBWYWxpZGF0ZSBzY2hlbWUsIGhvc3QgYW5kIGNyZWRlbnRpYWxzIGJlZm9yZSBhdXRob3JpemF0aW9uLgogICAgcmV0dXJuIHsKICAgICAgICAidmVyc2lvbl9pZCI6IHN0cih2ZXJzaW9uX2lkKSwgImZpbGVfaWQiOiBzZWxlY3RlZFsiaWQiXSwKICAgICAgICAic291cmNlX25hbWUiOiBzZWxlY3RlZFsibmFtZSJdLCAic291cmNlX3NoYTI1NiI6IGRpZ2VzdCwKICAgICAgICAiYmFzZV9tb2RlbCI6IHZlcnNpb24uZ2V0KCJiYXNlTW9kZWwiKSwgImRvd25sb2FkX3VybCI6IGRvd25sb2FkX3VybCwKICAgIH0KCgpkZWYgdmFsaWRhdGVfc2FmZXRlbnNvcnMocGF0aCk6CiAgICBzaXplID0gcGF0aC5zdGF0KCkuc3Rfc2l6ZQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgIHJhdyA9IHN0cmVhbS5yZWFkKDgpCiAgICAgICAgaWYgbGVuKHJhdykgIT0gODoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTm90IGEgc2FmZXRlbnNvcnMgZmlsZSIpCiAgICAgICAgbGVuZ3RoID0gc3RydWN0LnVucGFjaygiPFEiLCByYXcpWzBdCiAgICAgICAgaWYgbm90IDIgPD0gbGVuZ3RoIDw9IG1pbigxNl8wMDBfMDAwLCBzaXplIC0gOCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgc2FmZXRlbnNvcnMgaGVhZGVyIGxlbmd0aCIpCiAgICAgICAgaGVhZGVyID0ganNvbi5sb2FkcyhzdHJlYW0ucmVhZChsZW5ndGgpKQogICAgaWYgbm90IGlzaW5zdGFuY2UoaGVhZGVyLCBkaWN0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIHNhZmV0ZW5zb3JzIG1ldGFkYXRhIikKICAgIHRlbnNvcnMgPSB7azogdiBmb3IgaywgdiBpbiBoZWFkZXIuaXRlbXMoKSBpZiBrICE9ICJfX21ldGFkYXRhX18ifQogICAgaWYgbm90IHRlbnNvcnM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2FmZXRlbnNvcnMgZmlsZSBjb250YWlucyBubyB0ZW5zb3JzIikKICAgIGZvciB0ZW5zb3IgaW4gdGVuc29ycy52YWx1ZXMoKToKICAgICAgICBzdGFydCwgZW5kID0gdGVuc29yWyJkYXRhX29mZnNldHMiXQogICAgICAgIGlmIG5vdCAwIDw9IHN0YXJ0IDw9IGVuZCA8PSBzaXplIC0gOCAtIGxlbmd0aDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2FmZXRlbnNvcnMgZGF0YSBvZmZzZXRzIGV4Y2VlZCB0aGUgZmlsZSIpCiAgICByZXR1cm4gbGVuKHRlbnNvcnMpCgoKZGVmIGRvd25sb2FkX2xvcmEobWV0YWRhdGEsIHJvb3QsIHRva2VuLCBsb2dnZXIpOgogICAgcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkZXN0aW5hdGlvbiA9IHJvb3QgLyAobWV0YWRhdGFbInNvdXJjZV9zaGEyNTYiXSArICIuc2FmZXRlbnNvcnMiKQogICAgaWYgZGVzdGluYXRpb24uaXNfZmlsZSgpIGFuZCBjb250ZW50X2ZpbmdlcnByaW50KGRlc3RpbmF0aW9uKSA9PSBtZXRhZGF0YVsic291cmNlX3NoYTI1NiJdOgogICAgICAgIHZhbGlkYXRlX3NhZmV0ZW5zb3JzKGRlc3RpbmF0aW9uKQogICAgICAgIGxvZ2dlci5pbmZvKCJET1dOTE9BRCBSRVVTRUQ6IGNoZWNrc3VtIHZlcmlmaWVkIikKICAgICAgICByZXR1cm4gZGVzdGluYXRpb24KICAgIHBhcnRpYWwgPSBkZXN0aW5hdGlvbi53aXRoX3N1ZmZpeCgiLnBhcnQiKQogICAgb2Zmc2V0ID0gcGFydGlhbC5zdGF0KCkuc3Rfc2l6ZSBpZiBwYXJ0aWFsLmV4aXN0cygpIGVsc2UgMAogICAgIyBGdWxsIGV4cGVjdGVkIGNvbnRlbnQgaGFzaCBpcyBhdXRob3JpdGF0aXZlIGV2ZW4gaWYgdGhlIENETiBkb2VzIG5vdCBzdXBwbHkgRVRhZy4KICAgIGhlYWRlcnMgPSB7IlJhbmdlIjogZiJieXRlcz17b2Zmc2V0fS0ifSBpZiBvZmZzZXQgZWxzZSB7fQogICAgd2l0aCBvcGVuX3VybChtZXRhZGF0YVsiZG93bmxvYWRfdXJsIl0sIHRva2VuLCBoZWFkZXJzPWhlYWRlcnMpIGFzIHJlc3BvbnNlOgogICAgICAgIGlmIHJlc3BvbnNlLnN0YXR1cyA9PSAyMDY6CiAgICAgICAgICAgIGNvbnRlbnRfcmFuZ2UgPSByZXNwb25zZS5oZWFkZXJzLmdldCgiQ29udGVudC1SYW5nZSIsICIiKQogICAgICAgICAgICBpZiBub3QgY29udGVudF9yYW5nZS5zdGFydHN3aXRoKGYiYnl0ZXMge29mZnNldH0tIik6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkRvd25sb2FkIHNlcnZlciByZXR1cm5lZCBhbiB1bmV4cGVjdGVkIHJhbmdlIikKICAgICAgICAgICAgbW9kZSA9ICJhYiIKICAgICAgICBlbHNlOgogICAgICAgICAgICBvZmZzZXQgPSAwCiAgICAgICAgICAgIG1vZGUgPSAid2IiCiAgICAgICAgbG9nZ2VyLmluZm8oIkRPV05MT0FEIFNUQVJUOiB2ZXJzaW9uPSVzIGZpbGU9JXMgb2Zmc2V0PSVzIiwgbWV0YWRhdGFbInZlcnNpb25faWQiXSwgbWV0YWRhdGFbImZpbGVfaWQiXSwgb2Zmc2V0KQogICAgICAgIHRvdGFsID0gb2Zmc2V0CiAgICAgICAgbmV4dF9sb2cgPSB0b3RhbCArIDEyOCAqIDEwMjQgKiAxMDI0CiAgICAgICAgd2l0aCBwYXJ0aWFsLm9wZW4obW9kZSkgYXMgc3RyZWFtOgogICAgICAgICAgICB3aGlsZSBjaHVuayA6PSByZXNwb25zZS5yZWFkKDQgKiAxMDI0ICogMTAyNCk6CiAgICAgICAgICAgICAgICBzdHJlYW0ud3JpdGUoY2h1bmspCiAgICAgICAgICAgICAgICB0b3RhbCArPSBsZW4oY2h1bmspCiAgICAgICAgICAgICAgICBpZiB0b3RhbCA+PSBuZXh0X2xvZzoKICAgICAgICAgICAgICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJET1dOTE9BRDogJXMgTWlCIHNhdmVkIiwgdG90YWwgLy8gKDEwMjQgKiAxMDI0KSkKICAgICAgICAgICAgICAgICAgICBuZXh0X2xvZyA9IHRvdGFsICsgMTI4ICogMTAyNCAqIDEwMjQKICAgIGlmIGNvbnRlbnRfZmluZ2VycHJpbnQocGFydGlhbCkgIT0gbWV0YWRhdGFbInNvdXJjZV9zaGEyNTYiXToKICAgICAgICBwYXJ0aWFsLnJlbmFtZShwYXJ0aWFsLndpdGhfc3VmZml4KCIuc2hhMjU2LW1pc21hdGNoLSIgKyBzdHIoX19pbXBvcnRfXygndGltZScpLnRpbWVfbnMoKSkpKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkRvd25sb2FkIGNoZWNrc3VtIG1pc21hdGNoOyB0aGUgaW52YWxpZCBwYXJ0aWFsIHdhcyBwcmVzZXJ2ZWQgc2VwYXJhdGVseS4gUmVydW4gdG8gZG93bmxvYWQgYWdhaW4uIikKICAgIHZhbGlkYXRlX3NhZmV0ZW5zb3JzKHBhcnRpYWwpCiAgICBwYXJ0aWFsLnJlcGxhY2UoZGVzdGluYXRpb24pCiAgICBsb2dnZXIuaW5mbygiRE9XTkxPQUQgQ09NUExFVEU6IHZlcmlmaWVkIFNIQS0yNTYiKQogICAgcmV0dXJuIGRlc3RpbmF0aW9uCg==", "src/project_app/logging_utils.py": "IiIiUnVudGltZS1nbG9iYWwvcGVyLXJ1biBsb2dnaW5nIGFuZCBzdHJlYW1pbmcgc3VicHJvY2VzcyBleGVjdXRpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHJlCmltcG9ydCBzaGxleAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gdXJsbGliLnBhcnNlIGltcG9ydCB1cmxzcGxpdCwgdXJsdW5zcGxpdAppbXBvcnQgdXVpZAoKTE9HR0VSX05BTUUgPSAicHJvZHVjdGlvbl9jb2xhYiIKVVJMX1JFID0gcmUuY29tcGlsZShyImh0dHBzPzovL1teXHMnXCJdKyIpCgoKZGVmIHJlZGFjdF90ZXh0KHZhbHVlOiBvYmplY3QpIC0+IHN0cjoKICAgIGRlZiByZXBsYWNlKG1hdGNoOiByZS5NYXRjaFtzdHJdKSAtPiBzdHI6CiAgICAgICAgcGFyc2VkID0gdXJsc3BsaXQobWF0Y2guZ3JvdXAoMCkpCiAgICAgICAgaG9zdG5hbWUgPSBwYXJzZWQuaG9zdG5hbWUgb3IgIiIKICAgICAgICBpZiBwYXJzZWQucG9ydDoKICAgICAgICAgICAgaG9zdG5hbWUgKz0gZiI6e3BhcnNlZC5wb3J0fSIKICAgICAgICBuZXRsb2MgPSAoIioqKkAiIGlmIHBhcnNlZC51c2VybmFtZSBvciBwYXJzZWQucGFzc3dvcmQgZWxzZSAiIikgKyBob3N0bmFtZQogICAgICAgIHJldHVybiB1cmx1bnNwbGl0KChwYXJzZWQuc2NoZW1lLCBuZXRsb2MsIHBhcnNlZC5wYXRoLCAiIiwgIiIpKQoKICAgIHJldHVybiBVUkxfUkUuc3ViKHJlcGxhY2UsIHN0cih2YWx1ZSkpCgoKZGVmIHJ1bnRpbWVfbG9nX3BhdGgobG9nX3Jvb3Q6IFBhdGgpIC0+IFBhdGg6CiAgICBjb25maWd1cmVkID0gb3MuZW52aXJvbi5nZXQoIkNPTEFCX1JVTlRJTUVfTE9HIikKICAgIGlmIGNvbmZpZ3VyZWQ6CiAgICAgICAgcmV0dXJuIFBhdGgoY29uZmlndXJlZCkKICAgIHNlc3Npb25fZmlsZSA9IGxvZ19yb290LnBhcmVudCAvICIucnVudGltZV9zZXNzaW9uX2lkIgogICAgaWYgc2Vzc2lvbl9maWxlLmV4aXN0cygpOgogICAgICAgIHNlc3Npb25faWQgPSBzZXNzaW9uX2ZpbGUucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnN0cmlwKCkKICAgIGVsc2U6CiAgICAgICAgc2Vzc2lvbl9pZCA9IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLnN0cmZ0aW1lKCIlWSVtJWRUJUglTSVTWiIpICsgIi0iICsgdXVpZC51dWlkNCgpLmhleFs6OF0KICAgICAgICBzZXNzaW9uX2ZpbGUucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZXNzaW9uX2ZpbGUud3JpdGVfdGV4dChzZXNzaW9uX2lkICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJldHVybiBsb2dfcm9vdCAvIGYicnVudGltZV97c2Vzc2lvbl9pZH0ubG9nIgoKCmRlZiBjb25maWd1cmVfZ2xvYmFsX2xvZ2dlcihsb2dfcGF0aDogUGF0aCkgLT4gbG9nZ2luZy5Mb2dnZXI6CiAgICBsb2dfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoTE9HR0VSX05BTUUpCiAgICBsb2dnZXIuc2V0TGV2ZWwobG9nZ2luZy5ERUJVRykKICAgIGxvZ2dlci5wcm9wYWdhdGUgPSBGYWxzZQogICAgZm9ybWF0dGVyID0gbG9nZ2luZy5Gb3JtYXR0ZXIoIiUoYXNjdGltZSlzIHwgJShsZXZlbG5hbWUpcyB8ICUobWVzc2FnZSlzIikKCiAgICByZXNvbHZlZCA9IHN0cihsb2dfcGF0aC5yZXNvbHZlKCkpCiAgICBmb3IgaGFuZGxlciBpbiBsaXN0KGxvZ2dlci5oYW5kbGVycyk6CiAgICAgICAgaWYgZ2V0YXR0cihoYW5kbGVyLCAiX3Byb2R1Y3Rpb25fY29sYWJfZ2xvYmFsIiwgRmFsc2UpIGFuZCBnZXRhdHRyKGhhbmRsZXIsICJiYXNlRmlsZW5hbWUiLCBOb25lKSAhPSByZXNvbHZlZDoKICAgICAgICAgICAgbG9nZ2VyLnJlbW92ZUhhbmRsZXIoaGFuZGxlcikKICAgICAgICAgICAgaGFuZGxlci5jbG9zZSgpCiAgICBpZiBub3QgYW55KAogICAgICAgIGdldGF0dHIoaGFuZGxlciwgIl9wcm9kdWN0aW9uX2NvbGFiX2dsb2JhbCIsIEZhbHNlKQogICAgICAgIGFuZCBnZXRhdHRyKGhhbmRsZXIsICJiYXNlRmlsZW5hbWUiLCBOb25lKSA9PSByZXNvbHZlZAogICAgICAgIGZvciBoYW5kbGVyIGluIGxvZ2dlci5oYW5kbGVycwogICAgKToKICAgICAgICBmaWxlX2hhbmRsZXIgPSBsb2dnaW5nLkZpbGVIYW5kbGVyKGxvZ19wYXRoLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIGZpbGVfaGFuZGxlci5zZXRMZXZlbChsb2dnaW5nLkRFQlVHKQogICAgICAgIGZpbGVfaGFuZGxlci5zZXRGb3JtYXR0ZXIoZm9ybWF0dGVyKQogICAgICAgIGZpbGVfaGFuZGxlci5fcHJvZHVjdGlvbl9jb2xhYl9nbG9iYWwgPSBUcnVlICAjIHR5cGU6IGlnbm9yZVthdHRyLWRlZmluZWRdCiAgICAgICAgbG9nZ2VyLmFkZEhhbmRsZXIoZmlsZV9oYW5kbGVyKQoKICAgIGlmIG5vdCBhbnkoZ2V0YXR0cihoYW5kbGVyLCAiX3Byb2R1Y3Rpb25fY29sYWJfY29uc29sZSIsIEZhbHNlKSBmb3IgaGFuZGxlciBpbiBsb2dnZXIuaGFuZGxlcnMpOgogICAgICAgIGNvbnNvbGUgPSBsb2dnaW5nLlN0cmVhbUhhbmRsZXIoc3lzLnN0ZG91dCkKICAgICAgICBjb25zb2xlLnNldExldmVsKGxvZ2dpbmcuSU5GTykKICAgICAgICBjb25zb2xlLnNldEZvcm1hdHRlcihmb3JtYXR0ZXIpCiAgICAgICAgY29uc29sZS5fcHJvZHVjdGlvbl9jb2xhYl9jb25zb2xlID0gVHJ1ZSAgIyB0eXBlOiBpZ25vcmVbYXR0ci1kZWZpbmVkXQogICAgICAgIGxvZ2dlci5hZGRIYW5kbGVyKGNvbnNvbGUpCiAgICByZXR1cm4gbG9nZ2VyCgoKZGVmIGF0dGFjaF9ydW5fbG9nKGxvZ2dlcjogbG9nZ2luZy5Mb2dnZXIsIHJ1bl9sb2c6IFBhdGgpIC0+IGxvZ2dpbmcuSGFuZGxlcjoKICAgIHJ1bl9sb2cucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGhhbmRsZXIgPSBsb2dnaW5nLkZpbGVIYW5kbGVyKHJ1bl9sb2csIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBoYW5kbGVyLnNldExldmVsKGxvZ2dpbmcuREVCVUcpCiAgICBoYW5kbGVyLnNldEZvcm1hdHRlcihsb2dnaW5nLkZvcm1hdHRlcigiJShhc2N0aW1lKXMgfCAlKGxldmVsbmFtZSlzIHwgJShtZXNzYWdlKXMiKSkKICAgIGxvZ2dlci5hZGRIYW5kbGVyKGhhbmRsZXIpCiAgICByZXR1cm4gaGFuZGxlcgoKCmRlZiBydW5fbG9nZ2VkX2NvbW1hbmQoY29tbWFuZDogU2VxdWVuY2Vbc3RyXSwgbG9nZ2VyOiBsb2dnaW5nLkxvZ2dlciwgKiwgY3dkPU5vbmUsIGVudj1Ob25lKSAtPiBOb25lOgogICAgc2FmZV9jb21tYW5kID0gIiAiLmpvaW4oc2hsZXgucXVvdGUocmVkYWN0X3RleHQocGFydCkpIGZvciBwYXJ0IGluIGNvbW1hbmQpCiAgICBsb2dnZXIuaW5mbygiQ09NTUFORCBTVEFSVDogJXMiLCBzYWZlX2NvbW1hbmQpCiAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oCiAgICAgICAgW3N0cihwYXJ0KSBmb3IgcGFydCBpbiBjb21tYW5kXSwKICAgICAgICBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLAogICAgICAgIHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCwKICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgYnVmc2l6ZT0xLAogICAgICAgIGN3ZD1jd2QsCiAgICAgICAgZW52PWVudiwKICAgICkKICAgIGFzc2VydCBwcm9jZXNzLnN0ZG91dCBpcyBub3QgTm9uZQogICAgZm9yIGxpbmUgaW4gcHJvY2Vzcy5zdGRvdXQ6CiAgICAgICAgbG9nZ2VyLmluZm8oIiVzIiwgcmVkYWN0X3RleHQobGluZS5yc3RyaXAoIlxuIikpKQogICAgcmV0dXJuX2NvZGUgPSBwcm9jZXNzLndhaXQoKQogICAgZHVyYXRpb24gPSB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZAogICAgbG9nZ2VyLmluZm8oIkNPTU1BTkQgRU5EOiBleGl0X2NvZGU9JXMgZHVyYXRpb25fc2Vjb25kcz0lLjJmIiwgcmV0dXJuX2NvZGUsIGR1cmF0aW9uKQogICAgaWYgcmV0dXJuX2NvZGUgIT0gMDoKICAgICAgICByYWlzZSBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcihyZXR1cm5fY29kZSwgY29tbWFuZCkK", "src/project_app/orchestrator.py": "IiIiT25lIHN1Ym1pdHRlZC1qb2Igb3JjaGVzdHJhdG9yIHdpdGggZWFybHkgcnVuIGV2aWRlbmNlLCBzdGF0ZSwgYW5kIGRlbGl2ZXJ5LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0CmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaHV0aWwKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQppbXBvcnQgdXVpZAoKZnJvbSAuY29uZmlnIGltcG9ydCBKb2JDb25maWcsIGF0b21pY193cml0ZV9qc29uLCBzdWJtaXR0ZWRfam9iIGFzIG1ha2Vfc3VibWl0dGVkX2pvYiwgdXRjX25vdwpmcm9tIC5sb2dnaW5nX3V0aWxzIGltcG9ydCBhdHRhY2hfcnVuX2xvZywgY29uZmlndXJlX2dsb2JhbF9sb2dnZXIsIHJ1bnRpbWVfbG9nX3BhdGgKZnJvbSAucGF0aHMgaW1wb3J0IFByb2plY3RQYXRocwpmcm9tIC5yZXN1bHRzIGltcG9ydCBhbGxvY2F0ZV9vdXRwdXRfcGF0aCwgY3JlYXRlX21pbmltYWxfYnVuZGxlCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGlzX2NvbGFiX3J1bnRpbWUsIHByZXBhcmVfcnVudGltZQoKCmRlZiBlbnN1cmVfcGVyc2lzdGVudF9zdG9yYWdlKHBlcnNpc3RlbnRfcm9vdDogUGF0aCwgbG9nZ2VyKSAtPiBOb25lOgogICAgaWYgaXNfY29sYWJfcnVudGltZSgpIGFuZCBzdHIocGVyc2lzdGVudF9yb290KS5zdGFydHN3aXRoKCIvY29udGVudC8iKToKICAgICAgICBpZiBub3Qgc3RyKHBlcnNpc3RlbnRfcm9vdCkuc3RhcnRzd2l0aCgiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8iKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgICJDb2xhYiBwZXJzaXN0ZW50X3Jvb3QgdW5kZXIgL2NvbnRlbnQgbXVzdCBiZSBEcml2ZS1iYWNrZWQ7ICIKICAgICAgICAgICAgICAgICJ1c2UgL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8uLi4iCiAgICAgICAgICAgICkKICAgICAgICBkcml2ZV9yb290ID0gUGF0aCgiL2NvbnRlbnQvZHJpdmUiKQogICAgICAgIGlmIG5vdCAoZHJpdmVfcm9vdCAvICJNeURyaXZlIikuaXNfZGlyKCk6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJTVEFHRSBTVEFSVDogTU9VTlRfUEVSU0lTVEVOVF9TVE9SQUdFIikKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IGRyaXZlICAjIHR5cGU6IGlnbm9yZVtpbXBvcnQtbm90LWZvdW5kXQoKICAgICAgICAgICAgZHJpdmUubW91bnQoc3RyKGRyaXZlX3Jvb3QpKQogICAgICAgICAgICBsb2dnZXIuaW5mbygiU1RBR0UgQ09NUExFVEU6IE1PVU5UX1BFUlNJU1RFTlRfU1RPUkFHRSIpCiAgICBwZXJzaXN0ZW50X3Jvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKCmRlZiBfc3luY19ydW50aW1lX2xvZyhnbG9iYWxfbG9nOiBQYXRoLCBwYXRoczogUHJvamVjdFBhdGhzKSAtPiBQYXRoOgogICAgZGVzdGluYXRpb24gPSBwYXRocy5wZXJzaXN0ZW50X2xvZ19yb290IC8gZ2xvYmFsX2xvZy5uYW1lCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc2h1dGlsLmNvcHkyKGdsb2JhbF9sb2csIGRlc3RpbmF0aW9uKQogICAgcmV0dXJuIGRlc3RpbmF0aW9uCgoKZGVmIF9jb3B5X2ZpbmFsX291dHB1dChvdXRwdXRfcGF0aDogUGF0aCwgcGF0aHM6IFByb2plY3RQYXRocywgcnVuX2lkOiBzdHIpIC0+IFBhdGg6CiAgICBkZXN0aW5hdGlvbiA9IHBhdGhzLmV4cG9ydHNfcm9vdCAvIHJ1bl9pZCAvIG91dHB1dF9wYXRoLm5hbWUKICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzaHV0aWwuY29weTIob3V0cHV0X3BhdGgsIGRlc3RpbmF0aW9uKQogICAgY29tcGFuaW9uID0gb3V0cHV0X3BhdGgud2l0aF9zdWZmaXgoIi5jb252ZXJzaW9uLmpzb24iKQogICAgaWYgY29tcGFuaW9uLmlzX2ZpbGUoKToKICAgICAgICBzaHV0aWwuY29weTIoY29tcGFuaW9uLCBkZXN0aW5hdGlvbi53aXRoX3N1ZmZpeCgiLmNvbnZlcnNpb24uanNvbiIpKQogICAgcmV0dXJuIGRlc3RpbmF0aW9uCgoKZGVmIF9zZWVkX3J1bl9sb2cocnVuX2xvZzogUGF0aCkgLT4gTm9uZToKICAgIGJvb3RzdHJhcCA9IG9zLmVudmlyb24uZ2V0KCJDT0xBQl9TVUJNSVNTSU9OX0JPT1RTVFJBUF9MT0ciKQogICAgcnVuX2xvZy5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgaWYgYm9vdHN0cmFwIGFuZCBQYXRoKGJvb3RzdHJhcCkuaXNfZmlsZSgpOgogICAgICAgIHNodXRpbC5jb3B5Mihib290c3RyYXAsIHJ1bl9sb2cpCiAgICBlbHNlOgogICAgICAgIHJ1bl9sb2cudG91Y2goKQoKCmRlZiBfc3luY19ydW5fZXZpZGVuY2UoCiAgICBydW50aW1lX3J1bl9sb2c6IFBhdGgsCiAgICBydW50aW1lX21hbmlmZXN0OiBQYXRoLAogICAgcnVudGltZV9zdWJtaXR0ZWRfam9iOiBQYXRoLAogICAgcGVyc2lzdGVudF9ydW5fcm9vdDogUGF0aCwKKSAtPiB0dXBsZVtQYXRoLCBQYXRoLCBQYXRoXToKICAgIHBlcnNpc3RlbnRfcnVuX2xvZyA9IHBlcnNpc3RlbnRfcnVuX3Jvb3QgLyAibG9ncyIgLyAiZnVsbF9ydW4ubG9nIgogICAgcGVyc2lzdGVudF9tYW5pZmVzdCA9IHBlcnNpc3RlbnRfcnVuX3Jvb3QgLyAicnVuX21hbmlmZXN0Lmpzb24iCiAgICBwZXJzaXN0ZW50X3N1Ym1pdHRlZF9qb2IgPSBwZXJzaXN0ZW50X3J1bl9yb290IC8gInN1Ym1pdHRlZF9qb2IuanNvbiIKICAgIHBlcnNpc3RlbnRfcnVuX2xvZy5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc2h1dGlsLmNvcHkyKHJ1bnRpbWVfcnVuX2xvZywgcGVyc2lzdGVudF9ydW5fbG9nKQogICAgc2h1dGlsLmNvcHkyKHJ1bnRpbWVfbWFuaWZlc3QsIHBlcnNpc3RlbnRfbWFuaWZlc3QpCiAgICBzaHV0aWwuY29weTIocnVudGltZV9zdWJtaXR0ZWRfam9iLCBwZXJzaXN0ZW50X3N1Ym1pdHRlZF9qb2IpCiAgICByZXR1cm4gcGVyc2lzdGVudF9ydW5fbG9nLCBwZXJzaXN0ZW50X21hbmlmZXN0LCBwZXJzaXN0ZW50X3N1Ym1pdHRlZF9qb2IKCgpkZWYgX3dyaXRlX3BlcnNpc3RlbnRfbG9jYXRvcihjb25maWc6IEpvYkNvbmZpZykgLT4gUGF0aCB8IE5vbmU6CiAgICBpZiBub3Qgc3RyKGNvbmZpZy5wZXJzaXN0ZW50X3Jvb3QpLnN0YXJ0c3dpdGgoIi9jb250ZW50L2RyaXZlL015RHJpdmUvIik6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJlZ2lzdHJ5ID0gUGF0aCgiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9wcm9kdWN0aW9uX2NvbGFiX3JlZ2lzdHJ5IikgLyAoCiAgICAgICAgY29uZmlnLnByb2plY3RfaWQgKyAiLmpzb24iCiAgICApCiAgICBhdG9taWNfd3JpdGVfanNvbigKICAgICAgICByZWdpc3RyeSwKICAgICAgICB7CiAgICAgICAgICAgICJwcm9qZWN0X2lkIjogY29uZmlnLnByb2plY3RfaWQsCiAgICAgICAgICAgICJwZXJzaXN0ZW50X3Jvb3QiOiBzdHIoUGF0aChjb25maWcucGVyc2lzdGVudF9yb290KS5leHBhbmR1c2VyKCkpLAogICAgICAgICAgICAidXBkYXRlZF91dGMiOiB1dGNfbm93KCksCiAgICAgICAgfSwKICAgICkKICAgIHJldHVybiByZWdpc3RyeQoKCmRlZiBydW5fY29tcGxldGVfam9iKAogICAgY29uZmlnOiBKb2JDb25maWcsCiAgICBwcm9qZWN0X3Jvb3Q6IFBhdGgsCiAgICAqLAogICAgc3VibWl0dGVkX2pvYjogZGljdFtzdHIsIEFueV0gfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGNvbmZpZy52YWxpZGF0ZSgpCiAgICBmcm96ZW5fam9iID0gc3VibWl0dGVkX2pvYiBvciBtYWtlX3N1Ym1pdHRlZF9qb2IoY29uZmlnKQogICAgcnVudGltZV9yb290ID0gUGF0aChvcy5lbnZpcm9uLmdldCgiQ09MQUJfUlVOVElNRV9ST09UIiwgY29uZmlnLnJ1bnRpbWVfcm9vdCkpLmV4cGFuZHVzZXIoKQogICAgcnVudGltZV9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGdsb2JhbF9sb2cgPSBydW50aW1lX2xvZ19wYXRoKHJ1bnRpbWVfcm9vdCAvICJsb2dzIikKICAgIGxvZ2dlciA9IGNvbmZpZ3VyZV9nbG9iYWxfbG9nZ2VyKGdsb2JhbF9sb2cpCgogICAgcnVuX2lkID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0Yykuc3RyZnRpbWUoIiVZJW0lZFQlSCVNJVNaIikgKyAiLSIgKyB1dWlkLnV1aWQ0KCkuaGV4Wzo4XQogICAgcHJvdmlzaW9uYWxfcm9vdCA9IHJ1bnRpbWVfcm9vdCAvICJhY3RpdmVfcnVucyIgLyBydW5faWQKICAgIHJ1bl9sb2cgPSBwcm92aXNpb25hbF9yb290IC8gImxvZ3MiIC8gImZ1bGxfcnVuLmxvZyIKICAgIG1hbmlmZXN0ID0gcHJvdmlzaW9uYWxfcm9vdCAvICJydW5fbWFuaWZlc3QuanNvbiIKICAgIHN1Ym1pdHRlZF9wYXRoID0gcHJvdmlzaW9uYWxfcm9vdCAvICJzdWJtaXR0ZWRfam9iLmpzb24iCiAgICBfc2VlZF9ydW5fbG9nKHJ1bl9sb2cpCiAgICBydW5faGFuZGxlciA9IGF0dGFjaF9ydW5fbG9nKGxvZ2dlciwgcnVuX2xvZykKCiAgICBzdGF0ZTogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInJ1bl9pZCI6IHJ1bl9pZCwKICAgICAgICAic3VibWlzc2lvbl9pZCI6IGZyb3plbl9qb2JbInN1Ym1pc3Npb25faWQiXSwKICAgICAgICAic3RhdHVzIjogIlNVQk1JVFRFRCIsCiAgICAgICAgInN1Ym1pdHRlZF91dGMiOiBmcm96ZW5fam9iWyJzdWJtaXR0ZWRfdXRjIl0sCiAgICAgICAgImNvbmZpZ3VyYXRpb24iOiBhc2RpY3QoY29uZmlnKSwKICAgICAgICAicmVzb2x2ZWRfcHJvamVjdF9yZXZpc2lvbiI6IG9zLmVudmlyb24uZ2V0KAogICAgICAgICAgICAiQ09MQUJfUFJPSkVDVF9SRVNPTFZFRF9SRVZJU0lPTiIsIGNvbmZpZy5wcm9qZWN0X3JldmlzaW9uCiAgICAgICAgKSwKICAgICAgICAic3VibWl0dGVkX3Byb2plY3RfcmV2aXNpb24iOiBmcm96ZW5fam9iWyJzdWJtaXR0ZWRfcHJvamVjdF9yZXZpc2lvbiJdLAogICAgICAgICJnbG9iYWxfbG9nIjogc3RyKGdsb2JhbF9sb2cpLAogICAgICAgICJydW5fbG9nIjogc3RyKHJ1bl9sb2cpLAogICAgICAgICJzdWJtaXR0ZWRfam9iX3BhdGgiOiBzdHIoc3VibWl0dGVkX3BhdGgpLAogICAgICAgICJib290c3RyYXBfbG9nIjogb3MuZW52aXJvbi5nZXQoIkNPTEFCX1NVQk1JU1NJT05fQk9PVFNUUkFQX0xPRyIsICIiKSwKICAgIH0KICAgIGF0b21pY193cml0ZV9qc29uKHN1Ym1pdHRlZF9wYXRoLCBmcm96ZW5fam9iKQogICAgYXRvbWljX3dyaXRlX2pzb24obWFuaWZlc3QsIHN0YXRlKQogICAgcGF0aHM6IFByb2plY3RQYXRocyB8IE5vbmUgPSBOb25lCiAgICBwZXJzaXN0ZW50X3J1bl9yb290OiBQYXRoIHwgTm9uZSA9IE5vbmUKICAgIHBlcnNpc3RlbnRfcnVuX2xvZzogUGF0aCB8IE5vbmUgPSBOb25lCiAgICBwZXJzaXN0ZW50X21hbmlmZXN0OiBQYXRoIHwgTm9uZSA9IE5vbmUKICAgIHBlcnNpc3RlbnRfc3VibWl0dGVkX3BhdGg6IFBhdGggfCBOb25lID0gTm9uZQoKICAgIGRlZiBjaGVja3BvaW50KCkgLT4gTm9uZToKICAgICAgICBub25sb2NhbCBwZXJzaXN0ZW50X3J1bl9sb2csIHBlcnNpc3RlbnRfbWFuaWZlc3QsIHBlcnNpc3RlbnRfc3VibWl0dGVkX3BhdGgKICAgICAgICBhdG9taWNfd3JpdGVfanNvbihtYW5pZmVzdCwgc3RhdGUpCiAgICAgICAgaWYgcGVyc2lzdGVudF9ydW5fcm9vdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgcGVyc2lzdGVudF9ydW5fbG9nLAogICAgICAgICAgICAgICAgcGVyc2lzdGVudF9tYW5pZmVzdCwKICAgICAgICAgICAgICAgIHBlcnNpc3RlbnRfc3VibWl0dGVkX3BhdGgsCiAgICAgICAgICAgICkgPSBfc3luY19ydW5fZXZpZGVuY2UocnVuX2xvZywgbWFuaWZlc3QsIHN1Ym1pdHRlZF9wYXRoLCBwZXJzaXN0ZW50X3J1bl9yb290KQoKICAgIHRyeToKICAgICAgICBsb2dnZXIuaW5mbygiUlVOIFNUQVJUOiAlcyIsIHJ1bl9pZCkKICAgICAgICBzdGF0ZVsic3RhdHVzIl0gPSAiUkVTVE9SSU5HX1BFUlNJU1RFTlRfU1RBVEUiCiAgICAgICAgY2hlY2twb2ludCgpCiAgICAgICAgcGVyc2lzdGVudF9yb290ID0gUGF0aCgKICAgICAgICAgICAgb3MuZW52aXJvbi5nZXQoIkNPTEFCX1BFUlNJU1RFTlRfUk9PVCIsIGNvbmZpZy5wZXJzaXN0ZW50X3Jvb3QpCiAgICAgICAgKS5leHBhbmR1c2VyKCkKICAgICAgICBlbnN1cmVfcGVyc2lzdGVudF9zdG9yYWdlKHBlcnNpc3RlbnRfcm9vdCwgbG9nZ2VyKQogICAgICAgIHBhdGhzID0gUHJvamVjdFBhdGhzLmNyZWF0ZShwcm9qZWN0X3Jvb3QsIGNvbmZpZykKICAgICAgICBsb2NhdG9yID0gX3dyaXRlX3BlcnNpc3RlbnRfbG9jYXRvcihjb25maWcpCgogICAgICAgIHBlcnNpc3RlbnRfcnVuX3Jvb3QgPSBwYXRocy5ydW5zX3Jvb3QgLyBydW5faWQKICAgICAgICBzdGF0ZS51cGRhdGUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJydW50aW1lX3J1bl9sb2ciOiBzdHIocnVuX2xvZyksCiAgICAgICAgICAgICAgICAicnVuX2xvZyI6IHN0cihwZXJzaXN0ZW50X3J1bl9yb290IC8gImxvZ3MiIC8gImZ1bGxfcnVuLmxvZyIpLAogICAgICAgICAgICAgICAgInN1Ym1pdHRlZF9qb2JfcGF0aCI6IHN0cihwZXJzaXN0ZW50X3J1bl9yb290IC8gInN1Ym1pdHRlZF9qb2IuanNvbiIpLAogICAgICAgICAgICAgICAgInBlcnNpc3RlbnRfbG9jYXRvciI6IHN0cihsb2NhdG9yKSBpZiBsb2NhdG9yIGVsc2UgIiIsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICAgICAgY2hlY2twb2ludCgpCiAgICAgICAgYXRvbWljX3dyaXRlX2pzb24ocGF0aHMuc3RhdGVfcm9vdCAvICJwZW5kaW5nX2pvYi5qc29uIiwgZnJvemVuX2pvYikKCiAgICAgICAgc3RhdGVbInN0YXR1cyJdID0gIlBSRVBBUklOR19SVU5USU1FIgogICAgICAgIGNoZWNrcG9pbnQoKQogICAgICAgIHN0YXRlWyJydW50aW1lIl0gPSBwcmVwYXJlX3J1bnRpbWUocGF0aHMsIGNvbmZpZywgbG9nZ2VyKQogICAgICAgIGNoZWNrcG9pbnQoKQoKICAgICAgICBzdGF0ZVsic3RhdHVzIl0gPSAiVkFMSURBVElORyIKICAgICAgICBjaGVja3BvaW50KCkKICAgICAgICBpZiBub3QgY29uZmlnLmlucHV0X3VybDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2V0IGlucHV0X3VybCB0byBhIENpdml0YWkgTG9SQSBVUkwgaW4gdGhlIGNvbmZpZ3VyYXRpb24gZm9ybSIpCgogICAgICAgIHN0YXRlWyJzdGF0dXMiXSA9ICJQUk9DRVNTSU5HIgogICAgICAgIGNoZWNrcG9pbnQoKQogICAgICAgICMgSW1wb3J0IGRvbWFpbiBjb2RlIG9ubHkgYWZ0ZXIgdGhlIG1pc3NpbmcgZGVwZW5kZW5jeSBkZWx0YSBpcyBpbnN0YWxsZWQuCiAgICAgICAgZnJvbSAucGlwZWxpbmUgaW1wb3J0IGV4ZWN1dGVfcGlwZWxpbmUKCiAgICAgICAgb3V0cHV0X3BhdGggPSBhbGxvY2F0ZV9vdXRwdXRfcGF0aChwYXRocy5vdXRwdXRfcm9vdCwgY29uZmlnLm91dHB1dF9uYW1lKQogICAgICAgIGNvbnZlcnNpb24sIGNhY2hlX2V2ZW50ID0gZXhlY3V0ZV9waXBlbGluZSgKICAgICAgICAgICAgY29uZmlnLAogICAgICAgICAgICBvdXRwdXRfcGF0aCwKICAgICAgICAgICAgY2FjaGVfcm9vdD1wYXRocy5jYWNoZV9yb290LAogICAgICAgICAgICBsb2dnZXI9bG9nZ2VyLAogICAgICAgICkKICAgICAgICBzdGF0ZVsiY2FjaGUiXSA9IGNhY2hlX2V2ZW50CiAgICAgICAgc3RhdGVbImNvbnZlcnNpb24iXSA9IGNvbnZlcnNpb24KCiAgICAgICAgIyBTYXZlIHRoZSBmaW5hbCBvdXRwdXQgZmlyc3Q7IHBlcnNpc3RlbmNlIGFuZCBvcHRpb25hbCBicm93c2VyIGRlbGl2ZXJ5IGZvbGxvdy4KICAgICAgICBwZXJzaXN0ZW50X291dHB1dCA9IF9jb3B5X2ZpbmFsX291dHB1dChvdXRwdXRfcGF0aCwgcGF0aHMsIHJ1bl9pZCkKICAgICAgICBsb2dnZXIuaW5mbygiUlVOIENPTVBMRVRFOiBvdXRwdXQ9JXMiLCBvdXRwdXRfcGF0aCkKICAgICAgICBwZXJzaXN0ZW50X2dsb2JhbF9sb2cgPSBfc3luY19ydW50aW1lX2xvZyhnbG9iYWxfbG9nLCBwYXRocykKICAgICAgICBzdGF0ZS51cGRhdGUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJzdGF0dXMiOiAiQ09NUExFVEUiLAogICAgICAgICAgICAgICAgImNvbXBsZXRlZF91dGMiOiB1dGNfbm93KCksCiAgICAgICAgICAgICAgICAib3V0cHV0X3BhdGgiOiBzdHIob3V0cHV0X3BhdGgpLAogICAgICAgICAgICAgICAgInBlcnNpc3RlbnRfb3V0cHV0X3BhdGgiOiBzdHIocGVyc2lzdGVudF9vdXRwdXQpLAogICAgICAgICAgICAgICAgInBlcnNpc3RlbnRfZ2xvYmFsX2xvZyI6IHN0cihwZXJzaXN0ZW50X2dsb2JhbF9sb2cpLAogICAgICAgICAgICB9CiAgICAgICAgKQogICAgICAgIGNoZWNrcG9pbnQoKQogICAgICAgIGFzc2VydCAoCiAgICAgICAgICAgIHBlcnNpc3RlbnRfcnVuX2xvZyBpcyBub3QgTm9uZQogICAgICAgICAgICBhbmQgcGVyc2lzdGVudF9tYW5pZmVzdCBpcyBub3QgTm9uZQogICAgICAgICAgICBhbmQgcGVyc2lzdGVudF9zdWJtaXR0ZWRfcGF0aCBpcyBub3QgTm9uZQogICAgICAgICkKICAgICAgICBtaW5pbWFsX2J1bmRsZSA9IE5vbmUKICAgICAgICBwZXJzaXN0ZW50X21pbmltYWxfYnVuZGxlID0gTm9uZQogICAgICAgIGNoZWNrcG9pbnQoKQogICAgICAgIF9zeW5jX3J1bnRpbWVfbG9nKGdsb2JhbF9sb2csIHBhdGhzKQogICAgICAgIGFzc2VydCBwZXJzaXN0ZW50X3J1bl9sb2cgaXMgbm90IE5vbmUgYW5kIHBlcnNpc3RlbnRfbWFuaWZlc3QgaXMgbm90IE5vbmUKICAgICAgICByZXN1bHQgPSB7CiAgICAgICAgICAgICJydW5faWQiOiBydW5faWQsCiAgICAgICAgICAgICJzdGF0dXMiOiAiQ09NUExFVEUiLAogICAgICAgICAgICAib3V0cHV0X3BhdGgiOiBzdHIob3V0cHV0X3BhdGgpLAogICAgICAgICAgICAicGVyc2lzdGVudF9vdXRwdXRfcGF0aCI6IHN0cihwZXJzaXN0ZW50X291dHB1dCksCiAgICAgICAgICAgICJtYW5pZmVzdF9wYXRoIjogc3RyKHBlcnNpc3RlbnRfbWFuaWZlc3QpLAogICAgICAgICAgICAicnVuX2xvZyI6IHN0cihwZXJzaXN0ZW50X3J1bl9sb2cpLAogICAgICAgICAgICAicnVudGltZV9ydW5fbG9nIjogc3RyKHJ1bl9sb2cpLAogICAgICAgICAgICAiZ2xvYmFsX2xvZyI6IHN0cihnbG9iYWxfbG9nKSwKICAgICAgICB9CiAgICAgICAgaWYgbWluaW1hbF9idW5kbGUgaXMgbm90IE5vbmUgYW5kIHBlcnNpc3RlbnRfbWluaW1hbF9idW5kbGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgIm1pbmltYWxfYnVuZGxlX3BhdGgiOiBzdHIobWluaW1hbF9idW5kbGUpLAogICAgICAgICAgICAgICAgICAgICJwZXJzaXN0ZW50X21pbmltYWxfYnVuZGxlX3BhdGgiOiBzdHIocGVyc2lzdGVudF9taW5pbWFsX2J1bmRsZSksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gcmVzdWx0CiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgc3RhdGUudXBkYXRlKHsic3RhdHVzIjogIklOVEVSUlVQVEVEIiwgInVwZGF0ZWRfdXRjIjogdXRjX25vdygpfSkKICAgICAgICBsb2dnZXIuZXhjZXB0aW9uKCJSVU4gSU5URVJSVVBURUQiKQogICAgICAgIGNoZWNrcG9pbnQoKQogICAgICAgIGlmIHBhdGhzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBfc3luY19ydW50aW1lX2xvZyhnbG9iYWxfbG9nLCBwYXRocykKICAgICAgICByYWlzZQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgc3RhdGUudXBkYXRlKHsic3RhdHVzIjogIkZBSUxFRCIsICJ1cGRhdGVkX3V0YyI6IHV0Y19ub3coKSwgImVycm9yIjogcmVwcihleGMpfSkKICAgICAgICBsb2dnZXIuZXhjZXB0aW9uKCJSVU4gRkFJTEVEIikKICAgICAgICBjaGVja3BvaW50KCkKICAgICAgICBpZiBwYXRocyBpcyBub3QgTm9uZToKICAgICAgICAgICAgX3N5bmNfcnVudGltZV9sb2coZ2xvYmFsX2xvZywgcGF0aHMpCiAgICAgICAgcmFpc2UKICAgIGZpbmFsbHk6CiAgICAgICAgbG9nZ2VyLnJlbW92ZUhhbmRsZXIocnVuX2hhbmRsZXIpCiAgICAgICAgcnVuX2hhbmRsZXIuY2xvc2UoKQoKCmRlZiBsaXN0X3ByZXZpb3VzX3J1bnMoY29uZmlnOiBKb2JDb25maWcsIHByb2plY3Rfcm9vdDogUGF0aCwgKiwgbGltaXQ6IGludCA9IDUpIC0+IGxpc3RbZGljdFtzdHIsIHN0cl1dOgogICAgcGF0aHMgPSBQcm9qZWN0UGF0aHMuY3JlYXRlKHByb2plY3Rfcm9vdCwgY29uZmlnKQogICAgcmVjb3JkcyA9IFtdCiAgICBmb3IgbWFuaWZlc3QgaW4gc29ydGVkKHBhdGhzLnJ1bnNfcm9vdC5nbG9iKCIqL3J1bl9tYW5pZmVzdC5qc29uIiksIHJldmVyc2U9VHJ1ZSlbOmxpbWl0XToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBheWxvYWQgPSBqc29uLmxvYWRzKG1hbmlmZXN0LnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoeyJydW5faWQiOiBwYXlsb2FkWyJydW5faWQiXSwgInN0YXR1cyI6IHBheWxvYWRbInN0YXR1cyJdfSkKICAgICAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEtleUVycm9yKToKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoeyJydW5faWQiOiBtYW5pZmVzdC5wYXJlbnQubmFtZSwgInN0YXR1cyI6ICJVTlJFQURBQkxFIn0pCiAgICByZXR1cm4gcmVjb3Jkcwo=", "src/project_app/paths.py": "IiIiQ2VudHJhbGl6ZWQgcnVudGltZSwgcGVyc2lzdGVudCwgY2FjaGUsIHJ1biwgbG9nLCBhbmQgc2hhbGxvdyBvdXRwdXQgcGF0aHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSAuY29uZmlnIGltcG9ydCBKb2JDb25maWcKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBQcm9qZWN0UGF0aHM6CiAgICBwcm9qZWN0X3Jvb3Q6IFBhdGgKICAgIHJ1bnRpbWVfcm9vdDogUGF0aAogICAgcGVyc2lzdGVudF9yb290OiBQYXRoCiAgICBvdXRwdXRfcm9vdDogUGF0aAogICAgZ2xvYmFsX2xvZ19yb290OiBQYXRoCiAgICBwZXJzaXN0ZW50X2xvZ19yb290OiBQYXRoCiAgICBzdGF0ZV9yb290OiBQYXRoCiAgICBjYWNoZV9yb290OiBQYXRoCiAgICBydW5zX3Jvb3Q6IFBhdGgKICAgIGV4cG9ydHNfcm9vdDogUGF0aAoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGNyZWF0ZShjbHMsIHByb2plY3Rfcm9vdDogUGF0aCwgY29uZmlnOiBKb2JDb25maWcgfCBOb25lID0gTm9uZSkgLT4gIlByb2plY3RQYXRocyI6CiAgICAgICAgY29uZmlnID0gY29uZmlnIG9yIEpvYkNvbmZpZygpCiAgICAgICAgcnVudGltZV9yb290ID0gUGF0aChvcy5lbnZpcm9uLmdldCgiQ09MQUJfUlVOVElNRV9ST09UIiwgY29uZmlnLnJ1bnRpbWVfcm9vdCkpLmV4cGFuZHVzZXIoKQogICAgICAgIHBlcnNpc3RlbnRfcm9vdCA9IFBhdGgoCiAgICAgICAgICAgIG9zLmVudmlyb24uZ2V0KCJDT0xBQl9QRVJTSVNURU5UX1JPT1QiLCBjb25maWcucGVyc2lzdGVudF9yb290KQogICAgICAgICkuZXhwYW5kdXNlcigpCiAgICAgICAgb3V0cHV0X3Jvb3QgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJDT0xBQl9PVVRQVVRfUk9PVCIsICIvY29udGVudC9vdXRwdXRzIikpLmV4cGFuZHVzZXIoKQogICAgICAgIHBhdGhzID0gY2xzKAogICAgICAgICAgICBwcm9qZWN0X3Jvb3Q9cHJvamVjdF9yb290LAogICAgICAgICAgICBydW50aW1lX3Jvb3Q9cnVudGltZV9yb290LAogICAgICAgICAgICBwZXJzaXN0ZW50X3Jvb3Q9cGVyc2lzdGVudF9yb290LAogICAgICAgICAgICBvdXRwdXRfcm9vdD1vdXRwdXRfcm9vdCwKICAgICAgICAgICAgZ2xvYmFsX2xvZ19yb290PXJ1bnRpbWVfcm9vdCAvICJsb2dzIiwKICAgICAgICAgICAgcGVyc2lzdGVudF9sb2dfcm9vdD1wZXJzaXN0ZW50X3Jvb3QgLyAibG9ncyIsCiAgICAgICAgICAgIHN0YXRlX3Jvb3Q9cGVyc2lzdGVudF9yb290IC8gInN0YXRlIiwKICAgICAgICAgICAgY2FjaGVfcm9vdD1wZXJzaXN0ZW50X3Jvb3QgLyAiY2FjaGUiLAogICAgICAgICAgICBydW5zX3Jvb3Q9cGVyc2lzdGVudF9yb290IC8gInJ1bnMiLAogICAgICAgICAgICBleHBvcnRzX3Jvb3Q9cGVyc2lzdGVudF9yb290IC8gImV4cG9ydHMiLAogICAgICAgICkKICAgICAgICBmb3IgZGlyZWN0b3J5IGluICgKICAgICAgICAgICAgcGF0aHMucnVudGltZV9yb290LAogICAgICAgICAgICBwYXRocy5wZXJzaXN0ZW50X3Jvb3QsCiAgICAgICAgICAgIHBhdGhzLm91dHB1dF9yb290LAogICAgICAgICAgICBwYXRocy5nbG9iYWxfbG9nX3Jvb3QsCiAgICAgICAgICAgIHBhdGhzLnBlcnNpc3RlbnRfbG9nX3Jvb3QsCiAgICAgICAgICAgIHBhdGhzLnN0YXRlX3Jvb3QsCiAgICAgICAgICAgIHBhdGhzLmNhY2hlX3Jvb3QsCiAgICAgICAgICAgIHBhdGhzLnJ1bnNfcm9vdCwKICAgICAgICAgICAgcGF0aHMuZXhwb3J0c19yb290LAogICAgICAgICk6CiAgICAgICAgICAgIGRpcmVjdG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgcmV0dXJuIHBhdGhzCg==", "src/project_app/pipeline.py": "IiIiRG93bmxvYWQsIGNvbnZlcnQgYW5kIHBlcnNpc3Qgb25lIENpdml0YWkgTG9SQTsgbm8gZ2VuZXJhdGlvbiBvciB0cmFpbmluZy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCgpmcm9tIC5jYWNoZSBpbXBvcnQgSnNvblN0YWdlQ2FjaGUsIGNvbnRlbnRfZmluZ2VycHJpbnQKZnJvbSAuY29uZmlnIGltcG9ydCBhdG9taWNfd3JpdGVfanNvbgpmcm9tIC5jb252ZXJ0ZXIgaW1wb3J0IFVQU1RSRUFNX1JFVklTSU9OLCBCVUlMRF9SRVZJU0lPTiwgU09VUkNFX1BBVENIX1JFVklTSU9OLCBwcmVwYXJlX2NvbnZlcnRlciwgY29udmVydCwgdmFsaWRhdGVfY2hlY2twb2ludApmcm9tIC5kb3dubG9hZHMgaW1wb3J0IHJlc29sdmVfbG9yYSwgZG93bmxvYWRfbG9yYQoKCmRlZiBfc2VjcmV0KG5hbWUpOgogICAgaWYgbm90IG5hbWU6CiAgICAgICAgcmV0dXJuICIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmV0dXJuIG9zLmVudmlyb24uZ2V0KG5hbWUsICIiKQogICAgdHJ5OgogICAgICAgIHJldHVybiB1c2VyZGF0YS5nZXQobmFtZSkKICAgIGV4Y2VwdCB1c2VyZGF0YS5TZWNyZXROb3RGb3VuZEVycm9yOgogICAgICAgIHJldHVybiAiIgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvbGFiIHNlY3JldCBhY2Nlc3MgZmFpbGVkOyBhbGxvdyBub3RlYm9vayBhY2Nlc3Mgb3IgY2xlYXIgY2l2aXRhaV9zZWNyZXRfbmFtZSBmb3IgYSBwdWJsaWMgZG93bmxvYWQiKSBmcm9tIE5vbmUKCgpkZWYgZXhlY3V0ZV9waXBlbGluZShjb25maWcsIG91dHB1dF9wYXRoLCAqLCBjYWNoZV9yb290LCBsb2dnZXIpOgogICAgdG9rZW4gPSBfc2VjcmV0KGNvbmZpZy5jaXZpdGFpX3NlY3JldF9uYW1lKQogICAgbG9nZ2VyLmluZm8oIlNUQUdFIFNUQVJUOiBSRVNPTFZFX0NJVklUQUkiKQogICAgbWV0YWRhdGEgPSByZXNvbHZlX2xvcmEoY29uZmlnLmlucHV0X3VybCwgdG9rZW4pCiAgICBwZXJzaXN0ZW50ID0gY2FjaGVfcm9vdC5wYXJlbnQKICAgIHB1YmxpY19tZXRhZGF0YSA9IHtrZXk6IHZhbHVlIGZvciBrZXksIHZhbHVlIGluIG1ldGFkYXRhLml0ZW1zKCkgaWYga2V5ICE9ICJkb3dubG9hZF91cmwifQogICAgYXRvbWljX3dyaXRlX2pzb24ocGVyc2lzdGVudCAvICJpbnB1dHMiIC8gZiJ7bWV0YWRhdGFbJ3NvdXJjZV9zaGEyNTYnXX0uanNvbiIsIHB1YmxpY19tZXRhZGF0YSkKICAgIHNlbWFudGljX2lucHV0cyA9IHsKICAgICAgICAicHJvamVjdF9pZCI6IGNvbmZpZy5wcm9qZWN0X2lkLAogICAgICAgICJwcm9qZWN0X3JldmlzaW9uIjogb3MuZW52aXJvbi5nZXQoIkNPTEFCX1BST0pFQ1RfUkVTT0xWRURfUkVWSVNJT04iLCBjb25maWcucHJvamVjdF9yZXZpc2lvbiBvciAiZW1iZWRkZWQtdGVtcGxhdGUiKSwKICAgICAgICAic291cmNlX3NoYTI1NiI6IG1ldGFkYXRhWyJzb3VyY2Vfc2hhMjU2Il0sCiAgICAgICAgImNvbnZlcnRlcl9yZXZpc2lvbiI6IFVQU1RSRUFNX1JFVklTSU9OLAogICAgICAgICJjb252ZXJ0ZXJfc291cmNlX3BhdGNoIjogU09VUkNFX1BBVENIX1JFVklTSU9OLAogICAgICAgICJidWlsZF9yZXZpc2lvbiI6IEJVSUxEX1JFVklTSU9OLAogICAgICAgICJtb2RlbF92ZXJzaW9uIjogY29uZmlnLm1vZGVsX3ZlcnNpb24sCiAgICAgICAgInNjYWxlX2ZhY3RvciI6IGNvbmZpZy5zY2FsZV9mYWN0b3IsCiAgICB9CiAgICBzdGFnZV9jYWNoZSA9IEpzb25TdGFnZUNhY2hlKGNhY2hlX3Jvb3QsICJsb3JhX2NvbnZlcnNpb24iLCAib2ZmaWNpYWwtaW1wb3J0ZXItdjEiKQogICAgbG9va3VwID0gc3RhZ2VfY2FjaGUubG9hZChzZW1hbnRpY19pbnB1dHMpIGlmIGNvbmZpZy5yZXVzZV9jYWNoZSBlbHNlIE5vbmUKICAgIGNhY2hlZCA9IE5vbmUKICAgIGlmIGxvb2t1cCBpcyBub3QgTm9uZSBhbmQgbG9va3VwLnN0YXR1cyA9PSAiaGl0IjoKICAgICAgICBjYW5kaWRhdGUgPSBQYXRoKGxvb2t1cC52YWx1ZVsiY2hlY2twb2ludCJdKQogICAgICAgIGlmIGNhbmRpZGF0ZS5pc19maWxlKCkgYW5kIGNvbnRlbnRfZmluZ2VycHJpbnQoY2FuZGlkYXRlKSA9PSBsb29rdXAudmFsdWVbInNoYTI1NiJdOgogICAgICAgICAgICB2YWxpZGF0ZV9jaGVja3BvaW50KGNhbmRpZGF0ZSkKICAgICAgICAgICAgY2FjaGVkID0gY2FuZGlkYXRlCiAgICBpZiBjYWNoZWQ6CiAgICAgICAgbG9nZ2VyLmluZm8oIkNBQ0hFIEhJVDogb2ZmaWNpYWwgY29udmVyc2lvbiBjaGVja3N1bSB2ZXJpZmllZCIpCiAgICAgICAgY2FjaGVfZXZlbnQgPSBsb29rdXAuZXZlbnQoKQogICAgICAgICMgQSBwcmlvciBzdWNjZXNzZnVsIGNoZWNrcG9pbnQgaXMgc3VmZmljaWVudDsgZG8gbm90IHJlYnVpbGQgU3dpZnQuCiAgICAgICAgc2h1dGlsLmNvcHkyKGNhY2hlZCwgb3V0cHV0X3BhdGgpCiAgICBlbHNlOgogICAgICAgIGxvZ2dlci5pbmZvKCJDQUNIRSBNSVNTOiBjb252ZXJzaW9uIG5lZWRzIGV4ZWN1dGlvbiIpCiAgICAgICAgc291cmNlID0gZG93bmxvYWRfbG9yYShtZXRhZGF0YSwgcGVyc2lzdGVudCAvICJpbnB1dHMiLCB0b2tlbiwgbG9nZ2VyKQogICAgICAgIGxvZ2dlci5pbmZvKCJTVEFHRSBTVEFSVDogUFJFUEFSRV9DT05WRVJURVIiKQogICAgICAgIHJvb3QgPSBQYXRoKGNvbmZpZy5ydW50aW1lX3Jvb3QpIC8gImNvbnZlcnRlciIgLyAoVVBTVFJFQU1fUkVWSVNJT05bOjE2XSArICItIiArIEJVSUxEX1JFVklTSU9OKQogICAgICAgIGJpbmFyeSA9IHByZXBhcmVfY29udmVydGVyKHJvb3QsIGNvbmZpZy5idWlsZF9qb2JzLCBsb2dnZXIpCiAgICAgICAgbG9nZ2VyLmluZm8oIlNUQUdFIFNUQVJUOiBDT05WRVJUX0xPUkEiKQogICAgICAgIGNvbnZlcnQoYmluYXJ5LCBzb3VyY2UsIG91dHB1dF9wYXRoLCBjb25maWcsIGxvZ2dlcikKICAgICAgICBjaGVja3BvaW50X2hhc2ggPSBjb250ZW50X2ZpbmdlcnByaW50KG91dHB1dF9wYXRoKQogICAgICAgIGlmIGNvbmZpZy5yZXVzZV9jYWNoZToKICAgICAgICAgICAgIyBTYXZlIG9uZSByZXVzYWJsZSBjb3B5IG91dHNpZGUgcnVuIGRpcmVjdG9yaWVzLiBObyBhZGRpdGlvbmFsIHdlaWdodCBaSVAuCiAgICAgICAgICAgIGNhY2hlX2NoZWNrcG9pbnQgPSBjYWNoZV9yb290IC8gImNoZWNrcG9pbnRzIiAvIChjaGVja3BvaW50X2hhc2ggKyAiLmNrcHQiKQogICAgICAgICAgICBjYWNoZV9jaGVja3BvaW50LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIHRlbXBvcmFyeSA9IGNhY2hlX2NoZWNrcG9pbnQud2l0aF9zdWZmaXgoIi5ja3B0LnRtcCIpCiAgICAgICAgICAgIHNodXRpbC5jb3B5MihvdXRwdXRfcGF0aCwgdGVtcG9yYXJ5KQogICAgICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShjYWNoZV9jaGVja3BvaW50KQogICAgICAgICAgICB2YWx1ZSA9IHsiY2hlY2twb2ludCI6IHN0cihjYWNoZV9jaGVja3BvaW50KSwgInNoYTI1NiI6IGNoZWNrcG9pbnRfaGFzaH0KICAgICAgICAgICAgY2FjaGVfZXZlbnQgPSBzdGFnZV9jYWNoZS5zYXZlKHNlbWFudGljX2lucHV0cywgdmFsdWUpLmV2ZW50KCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBjYWNoZV9ldmVudCA9IHsic3RhdHVzIjogImRpc2FibGVkIiwgInJlYXNvbiI6ICJyZXVzZV9kaXNhYmxlZCIsICJrZXkiOiBzdGFnZV9jYWNoZS5rZXkoc2VtYW50aWNfaW5wdXRzKX0KICAgIHZhbGlkYXRlX2NoZWNrcG9pbnQob3V0cHV0X3BhdGgpCiAgICByZWNvcmQgPSB7CiAgICAgICAgKipwdWJsaWNfbWV0YWRhdGEsCiAgICAgICAgIm91dHB1dF9zaGEyNTYiOiBjb250ZW50X2ZpbmdlcnByaW50KG91dHB1dF9wYXRoKSwKICAgICAgICAib3V0cHV0X2J5dGVzIjogb3V0cHV0X3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgImNvbnZlcnRlcl9yZXZpc2lvbiI6IFVQU1RSRUFNX1JFVklTSU9OLAogICAgICAgICJjb252ZXJ0ZXJfc291cmNlX3BhdGNoIjogU09VUkNFX1BBVENIX1JFVklTSU9OLAogICAgICAgICJmb3JjZWRfbW9kZWxfdmVyc2lvbiI6IGNvbmZpZy5tb2RlbF92ZXJzaW9uIG9yIE5vbmUsCiAgICAgICAgInNjYWxlX2ZhY3RvciI6IGNvbmZpZy5zY2FsZV9mYWN0b3IsCiAgICAgICAgImNhY2hlIjogY2FjaGVfZXZlbnQsCiAgICAgICAgInZhbGlkYXRpb24iOiAiU1FMaXRlIGludGVncml0eSBvbmx5OyBydW50aW1lIExvUkEgYXBwbGljYXRpb24gaXMgbm90IHZlcmlmaWVkIiwKICAgIH0KICAgIGF0b21pY193cml0ZV9qc29uKG91dHB1dF9wYXRoLndpdGhfc3VmZml4KCIuY29udmVyc2lvbi5qc29uIiksIHJlY29yZCkKICAgIGxvZ2dlci5pbmZvKCJTVEFHRSBDT01QTEVURTogQ09OVkVSVF9MT1JBIG91dHB1dD0lcyIsIG91dHB1dF9wYXRoKQogICAgcmV0dXJuIHJlY29yZCwgY2FjaGVfZXZlbnQK", "src/project_app/results.py": "IiIiU2hhbGxvdyBvdXRwdXQgYWxsb2NhdGlvbiwgcGVyc2lzdGVudCBjb3BpZXMsIHByZXZpZXdzLCBhbmQgcmVzaWxpZW50IGRvd25sb2Fkcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CmltcG9ydCB6aXBmaWxlCgoKZGVmIGFsbG9jYXRlX291dHB1dF9wYXRoKG91dHB1dF9yb290OiBQYXRoLCBvdXRwdXRfbmFtZTogc3RyKSAtPiBQYXRoOgogICAgb3V0cHV0X3Jvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmVxdWVzdGVkID0gb3V0cHV0X3Jvb3QgLyBvdXRwdXRfbmFtZQogICAgaWYgbm90IHJlcXVlc3RlZC5leGlzdHMoKToKICAgICAgICByZXR1cm4gcmVxdWVzdGVkCiAgICBzdGVtLCBzdWZmaXggPSByZXF1ZXN0ZWQuc3RlbSwgcmVxdWVzdGVkLnN1ZmZpeAogICAgaW5kZXggPSAyCiAgICB3aGlsZSBUcnVlOgogICAgICAgIGNhbmRpZGF0ZSA9IG91dHB1dF9yb290IC8gZiJ7c3RlbX0te2luZGV4fXtzdWZmaXh9IgogICAgICAgIGlmIG5vdCBjYW5kaWRhdGUuZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBjYW5kaWRhdGUKICAgICAgICBpbmRleCArPSAxCgoKZGVmIGNyZWF0ZV9taW5pbWFsX2J1bmRsZSgKICAgIG91dHB1dF9wYXRoOiBQYXRoLAogICAgbWFuaWZlc3RfcGF0aDogUGF0aCwKICAgIHN1Ym1pdHRlZF9qb2JfcGF0aDogUGF0aCwKICAgIHJ1bl9sb2dfcGF0aDogUGF0aCwKKSAtPiBQYXRoOgogICAgYnVuZGxlX3BhdGggPSBhbGxvY2F0ZV9vdXRwdXRfcGF0aCgKICAgICAgICBvdXRwdXRfcGF0aC5wYXJlbnQsCiAgICAgICAgZiJ7b3V0cHV0X3BhdGguc3RlbX0tbWluaW1hbC56aXAiLAogICAgKQogICAgdGVtcG9yYXJ5ID0gYnVuZGxlX3BhdGgud2l0aF9zdWZmaXgoYnVuZGxlX3BhdGguc3VmZml4ICsgIi50bXAiKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUodGVtcG9yYXJ5LCAidyIsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBwYXRoLCBhcmNoaXZlX25hbWUgaW4gKAogICAgICAgICAgICAob3V0cHV0X3BhdGgsIG91dHB1dF9wYXRoLm5hbWUpLAogICAgICAgICAgICAobWFuaWZlc3RfcGF0aCwgInJ1bl9tYW5pZmVzdC5qc29uIiksCiAgICAgICAgICAgIChzdWJtaXR0ZWRfam9iX3BhdGgsICJzdWJtaXR0ZWRfam9iLmpzb24iKSwKICAgICAgICAgICAgKHJ1bl9sb2dfcGF0aCwgImxvZ3MvZnVsbF9ydW4ubG9nIiksCiAgICAgICAgKToKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShwYXRoLCBhcmNuYW1lPWFyY2hpdmVfbmFtZSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKGJ1bmRsZV9wYXRoKQogICAgcmV0dXJuIGJ1bmRsZV9wYXRoCgoKZGVmIF9zaG93X2ZpbGVfbGluayhwYXRoOiBQYXRoLCBsYWJlbDogc3RyKSAtPiBOb25lOgogICAgcHJpbnQoZiJ7bGFiZWx9OiB7cGF0aH0iKQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIGZyb20gSVB5dGhvbi5kaXNwbGF5IGltcG9ydCBGaWxlTGluaywgZGlzcGxheQoKICAgICAgICBkaXNwbGF5KEZpbGVMaW5rKHN0cihwYXRoKSwgcmVzdWx0X2h0bWxfcHJlZml4PWYie2xhYmVsfTogIikpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBwcmludChmIntsYWJlbH0gbGluayB1bmF2YWlsYWJsZSBpbiB0aGlzIGZyb250ZW5kOiB7ZXhjfSIpCgoKZGVmIHNob3dfcmVzdWx0X3BhbmVsKHJ1bl9yZXN1bHQ6IGRpY3Rbc3RyLCBBbnldLCAqLCBhdXRvX2Rvd25sb2FkOiBib29sKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIG91dHB1dF9wYXRoID0gUGF0aChydW5fcmVzdWx0WyJvdXRwdXRfcGF0aCJdKQogICAgcHJpbnQoIlJVTiBDT01QTEVURSIpCiAgICBfc2hvd19maWxlX2xpbmsob3V0cHV0X3BhdGgsICJGaW5hbCBvdXRwdXQiKQogICAgaWYgcnVuX3Jlc3VsdC5nZXQoInBlcnNpc3RlbnRfb3V0cHV0X3BhdGgiKToKICAgICAgICBwcmludChmIlBlcnNpc3RlbnQgY29weToge3J1bl9yZXN1bHRbJ3BlcnNpc3RlbnRfb3V0cHV0X3BhdGgnXX0iKQogICAgaWYgcnVuX3Jlc3VsdC5nZXQoIm1pbmltYWxfYnVuZGxlX3BhdGgiKToKICAgICAgICBfc2hvd19maWxlX2xpbmsoUGF0aChydW5fcmVzdWx0WyJtaW5pbWFsX2J1bmRsZV9wYXRoIl0pLCAiTWluaW1hbCBidW5kbGUiKQogICAgX3Nob3dfZmlsZV9saW5rKFBhdGgocnVuX3Jlc3VsdFsibWFuaWZlc3RfcGF0aCJdKSwgIlJ1biBtYW5pZmVzdCIpCiAgICBfc2hvd19maWxlX2xpbmsoUGF0aChydW5fcmVzdWx0WyJydW5fbG9nIl0pLCAiRnVsbCBydW4gbG9nIikKCiAgICBtYW51YWxfbGluayA9IHN0cihvdXRwdXRfcGF0aCkKCiAgICBpZiBvdXRwdXRfcGF0aC5zdWZmaXgubG93ZXIoKSA9PSAiLmpzb24iIGFuZCBvdXRwdXRfcGF0aC5zdGF0KCkuc3Rfc2l6ZSA8PSAxXzAwMF8wMDA6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcmV2aWV3ID0ganNvbi5sb2FkcyhvdXRwdXRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgICAgIHByaW50KCJQcmV2aWV3OiIpCiAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMocHJldmlldywgaW5kZW50PTIpWzo0MDAwXSkKICAgICAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICBwYXNzCgogICAgZG93bmxvYWRfc3RhdHVzID0gImRpc2FibGVkIgogICAgaWYgYXV0b19kb3dubG9hZDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBmaWxlcyAgIyB0eXBlOiBpZ25vcmVbaW1wb3J0LW5vdC1mb3VuZF0KCiAgICAgICAgICAgIGZpbGVzLmRvd25sb2FkKHN0cihvdXRwdXRfcGF0aCkpCiAgICAgICAgICAgIGRvd25sb2FkX3N0YXR1cyA9ICJyZXF1ZXN0ZWQiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIGRvd25sb2FkX3N0YXR1cyA9ICJibG9ja2VkIgogICAgICAgICAgICBwcmludChmIkF1dG9tYXRpYyBkb3dubG9hZCBjb3VsZCBub3Qgc3RhcnQ7IHRoZSBzYXZlZCByZXN1bHQgaXMgc3RpbGwgdmFsaWQ6IHtleGN9IikKICAgIHJldHVybiB7Im1hbnVhbF9saW5rIjogbWFudWFsX2xpbmssICJhdXRvbWF0aWNfZG93bmxvYWQiOiBkb3dubG9hZF9zdGF0dXN9Cg==", "src/project_app/runtime.py": "IiIiRnJlc2gtcnVudGltZSBib290c3RyYXAgd2l0aCBmaW5nZXJwcmludGVkIHJlYWRpbmVzcyBhbmQgZGVwZW5kZW5jeSBkZWx0YSBpbnN0YWxscy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmZyb20gaW1wb3J0bGliIGltcG9ydCBtZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3lzCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gLmNvbmZpZyBpbXBvcnQgSm9iQ29uZmlnLCBhdG9taWNfd3JpdGVfanNvbiwgdXRjX25vdwpmcm9tIC5sb2dnaW5nX3V0aWxzIGltcG9ydCBydW5fbG9nZ2VkX2NvbW1hbmQKZnJvbSAucGF0aHMgaW1wb3J0IFByb2plY3RQYXRocwoKX05BTUVfUkUgPSByZS5jb21waWxlKHIiXihbQS1aYS16MC05XVtBLVphLXowLTlfLi1dKikiKQoKCmRlZiBpc19jb2xhYl9ydW50aW1lKCkgLT4gYm9vbDoKICAgIHRyeToKICAgICAgICBpbXBvcnQgZ29vZ2xlLmNvbGFiICAjIHR5cGU6IGlnbm9yZVtpbXBvcnQtbm90LWZvdW5kXSAgIyBub3FhOiBGNDAxCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gVHJ1ZQoKCmRlZiBlbnZpcm9ubWVudF9maW5nZXJwcmludChwcm9qZWN0X3Jvb3Q6IFBhdGgsIGNvbmZpZzogSm9iQ29uZmlnKSAtPiB0dXBsZVtzdHIsIGRpY3Rbc3RyLCBBbnldXToKICAgIGxvY2tfcGF0aCA9IHByb2plY3Rfcm9vdCAvICJyZXF1aXJlbWVudHMubG9jayIKICAgIGxvY2tfYnl0ZXMgPSBsb2NrX3BhdGgucmVhZF9ieXRlcygpIGlmIGxvY2tfcGF0aC5leGlzdHMoKSBlbHNlIGIiIgogICAgcGF5bG9hZCA9IHsKICAgICAgICAicHl0aG9uIjogZiJ7c3lzLnZlcnNpb25faW5mby5tYWpvcn0ue3N5cy52ZXJzaW9uX2luZm8ubWlub3J9IiwKICAgICAgICAicnVudGltZV9wcm9maWxlIjogImRlZmF1bHQiLAogICAgICAgICJwcm9qZWN0X3JldmlzaW9uIjogb3MuZW52aXJvbi5nZXQoCiAgICAgICAgICAgICJDT0xBQl9QUk9KRUNUX1JFU09MVkVEX1JFVklTSU9OIiwgY29uZmlnLnByb2plY3RfcmV2aXNpb24KICAgICAgICApLAogICAgICAgICJkZXBlbmRlbmN5X2xvY2tfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYobG9ja19ieXRlcykuaGV4ZGlnZXN0KCksCiAgICB9CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigKICAgICAgICBqc29uLmR1bXBzKHBheWxvYWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgiLCIsICI6IikpLmVuY29kZSgidXRmLTgiKQogICAgKS5oZXhkaWdlc3QoKVs6MTZdCiAgICByZXR1cm4gZGlnZXN0LCBwYXlsb2FkCgoKZGVmIF9yZXF1aXJlbWVudF9pc19zYXRpc2ZpZWQocmVxdWlyZW1lbnQ6IHN0cikgLT4gYm9vbDoKICAgIG1hdGNoID0gX05BTUVfUkUubWF0Y2gocmVxdWlyZW1lbnQpCiAgICBpZiBub3QgbWF0Y2g6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIHJlcXVpcmVtZW50IGxpbmU6IHtyZXF1aXJlbWVudCFyfSIpCiAgICBuYW1lID0gbWF0Y2guZ3JvdXAoMSkKICAgIHRyeToKICAgICAgICBpbnN0YWxsZWQgPSBtZXRhZGF0YS52ZXJzaW9uKG5hbWUpCiAgICBleGNlcHQgbWV0YWRhdGEuUGFja2FnZU5vdEZvdW5kRXJyb3I6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICB0cnk6CiAgICAgICAgZnJvbSBwYWNrYWdpbmcucmVxdWlyZW1lbnRzIGltcG9ydCBSZXF1aXJlbWVudAoKICAgICAgICBwYXJzZWQgPSBSZXF1aXJlbWVudChyZXF1aXJlbWVudCkKICAgICAgICByZXR1cm4gbm90IHBhcnNlZC5zcGVjaWZpZXIgb3IgcGFyc2VkLnNwZWNpZmllci5jb250YWlucyhpbnN0YWxsZWQsIHByZXJlbGVhc2VzPVRydWUpCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgZXhhY3QgPSByZXF1aXJlbWVudC5zcGxpdCgiPT0iLCAxKQogICAgICAgIGlmIGxlbihleGFjdCkgPT0gMToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgIlRoZSBwYWNrYWdpbmcgcGFja2FnZSBpcyByZXF1aXJlZCB0byBldmFsdWF0ZSBub24tZXhhY3QgZGVwZW5kZW5jeSBjb25zdHJhaW50cyIKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBpbnN0YWxsZWQgPT0gZXhhY3RbMV0uc3RyaXAoKQoKCmRlZiBkZXBlbmRlbmN5X2RlbHRhKGxvY2tfcGF0aDogUGF0aCkgLT4gbGlzdFtzdHJdOgogICAgaWYgbm90IGxvY2tfcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gW10KICAgIHJlcXVpcmVtZW50cyA9IFtdCiAgICBmb3IgcmF3IGluIGxvY2tfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIGxpbmUgPSByYXcuc3BsaXQoIiMiLCAxKVswXS5zdHJpcCgpCiAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgcmVxdWlyZW1lbnRzLmFwcGVuZChsaW5lKQogICAgcmV0dXJuIFtyZXF1aXJlbWVudCBmb3IgcmVxdWlyZW1lbnQgaW4gcmVxdWlyZW1lbnRzIGlmIG5vdCBfcmVxdWlyZW1lbnRfaXNfc2F0aXNmaWVkKHJlcXVpcmVtZW50KV0KCgpkZWYgcHJlcGFyZV9ydW50aW1lKHBhdGhzOiBQcm9qZWN0UGF0aHMsIGNvbmZpZzogSm9iQ29uZmlnLCBsb2dnZXIpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgZmluZ2VycHJpbnQsIGZpbmdlcnByaW50X3BheWxvYWQgPSBlbnZpcm9ubWVudF9maW5nZXJwcmludChwYXRocy5wcm9qZWN0X3Jvb3QsIGNvbmZpZykKICAgIG1hcmtlciA9IHBhdGhzLnJ1bnRpbWVfcm9vdCAvIGYiLnByb2R1Y3Rpb25fY29sYWJfe2ZpbmdlcnByaW50fS5yZWFkeSIKICAgIGlmIG1hcmtlci5leGlzdHMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIG1hcmtlcl9wYXlsb2FkID0ganNvbi5sb2FkcyhtYXJrZXIucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvciwgVHlwZUVycm9yKToKICAgICAgICAgICAgbWFya2VyX3BheWxvYWQgPSBOb25lCiAgICAgICAgaWYgaXNpbnN0YW5jZShtYXJrZXJfcGF5bG9hZCwgZGljdCkgYW5kIG1hcmtlcl9wYXlsb2FkLmdldCgiZmluZ2VycHJpbnQiKSA9PSBmaW5nZXJwcmludDoKICAgICAgICAgICAgbG9nZ2VyLmluZm8oIlNUQUdFIFJFVVNFRDogUFJFUEFSRV9SVU5USU1FIGZpbmdlcnByaW50PSVzIiwgZmluZ2VycHJpbnQpCiAgICAgICAgICAgIHJldHVybiB7ImZpbmdlcnByaW50IjogZmluZ2VycHJpbnQsICJyZXVzZWQiOiBUcnVlLCAiaW5zdGFsbGVkIjogW119CiAgICAgICAgbG9nZ2VyLndhcm5pbmcoIklnbm9yaW5nIGludmFsaWQgcnVudGltZS1yZWFkeSBtYXJrZXI6ICVzIiwgbWFya2VyKQoKICAgIGxvZ2dlci5pbmZvKCJTVEFHRSBTVEFSVDogUFJFUEFSRV9SVU5USU1FIGZpbmdlcnByaW50PSVzIiwgZmluZ2VycHJpbnQpCiAgICBzaHV0aWwucm10cmVlKCIvY29udGVudC9zYW1wbGVfZGF0YSIsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKICAgIGlmIGlzX2NvbGFiX3J1bnRpbWUoKSBhbmQgc2h1dGlsLndoaWNoKCJudnRvcCIpIGlzIE5vbmU6CiAgICAgICAgcnVuX2xvZ2dlZF9jb21tYW5kKFsiYXB0LWdldCIsICJpbnN0YWxsIiwgIi15IiwgIm52dG9wIl0sIGxvZ2dlcikKCiAgICBtaXNzaW5nID0gZGVwZW5kZW5jeV9kZWx0YShwYXRocy5wcm9qZWN0X3Jvb3QgLyAicmVxdWlyZW1lbnRzLmxvY2siKQogICAgaWYgbWlzc2luZzoKICAgICAgICBsb2dnZXIuaW5mbygiSW5zdGFsbGluZyBtaXNzaW5nIG9yIGluY29tcGF0aWJsZSBkZXBlbmRlbmNpZXM6ICVzIiwgIiwgIi5qb2luKG1pc3NpbmcpKQogICAgICAgIHJ1bl9sb2dnZWRfY29tbWFuZChbc3lzLmV4ZWN1dGFibGUsICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICptaXNzaW5nXSwgbG9nZ2VyKQogICAgZWxzZToKICAgICAgICBsb2dnZXIuaW5mbygiUHl0aG9uIGRlcGVuZGVuY3kgZGVsdGE6IG5vbmUiKQoKICAgIG1hbmlmZXN0ID0gewogICAgICAgICoqZmluZ2VycHJpbnRfcGF5bG9hZCwKICAgICAgICAiZmluZ2VycHJpbnQiOiBmaW5nZXJwcmludCwKICAgICAgICAicHJlcGFyZWRfdXRjIjogdXRjX25vdygpLAogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgImluc3RhbGxlZF9kZWx0YSI6IG1pc3NpbmcsCiAgICB9CiAgICBhdG9taWNfd3JpdGVfanNvbihwYXRocy5zdGF0ZV9yb290IC8gImVudmlyb25tZW50X21hbmlmZXN0Lmpzb24iLCBtYW5pZmVzdCkKICAgIGF0b21pY193cml0ZV9qc29uKG1hcmtlciwgbWFuaWZlc3QpCiAgICBsb2dnZXIuaW5mbygiU1RBR0UgQ09NUExFVEU6IFBSRVBBUkVfUlVOVElNRSBpbnN0YWxsZWQ9JXMiLCBsZW4obWlzc2luZykpCiAgICByZXR1cm4geyJmaW5nZXJwcmludCI6IGZpbmdlcnByaW50LCAicmV1c2VkIjogRmFsc2UsICJpbnN0YWxsZWQiOiBtaXNzaW5nfQo=", "src/project_app/support/__init__.py": "IiIiTmVzdGVkIHBhY2thZ2UgcmV0YWluZWQgYnkgdGhlIGRldGVybWluaXN0aWMgZW1iZWRkZWQgYm9vdHN0cmFwLiIiIgo=", "src/project_app/support/identity.py": "IiIiU21hbGwgbmVzdGVkLW1vZHVsZSBmaXh0dXJlIHVzZWQgYnkgZW1iZWRkZWQtYm9vdHN0cmFwIHJlZ3Jlc3Npb24gdGVzdHMuIiIiCgpORVNURURfQk9PVFNUUkFQX0ZJWFRVUkUgPSAiYXZhaWxhYmxlIgo="}  # generated
_EMBEDDED_PROJECT_REVISION = '32562b58e1e554f3ee1817ac46631110e7f11d84097379f7086b09dd60b2788a'  # generated
_EMBEDDED_PROJECT_ID = 'lora-converter'  # generated
_EMBEDDED_DEFAULT_RUNTIME_ROOT = '/content/production_colab/lora-converter'  # generated
_EMBEDDED_DEFAULT_PERSISTENT_ROOT = '/content/drive/MyDrive/production_colab/lora-converter'  # generated
_URL_RE = re.compile(r"https?://[^\s'\"]+")
_EXECUTION_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
)
_BUNDLE_EXCLUDED_DIRECTORIES = {"__pycache__"}
_BUNDLE_EXCLUDED_SUFFIXES = {".pyc", ".pyo"}


def _atomic_write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)


def _redact_argument(value):
    text = str(value)
    if "://" not in text:
        return text
    parsed = urlsplit(text)
    hostname = parsed.hostname or ""
    if parsed.port:
        hostname += f":{parsed.port}"
    netloc = ("***@" if parsed.username or parsed.password else "") + hostname
    return urlunsplit((parsed.scheme, netloc, parsed.path, "", ""))


def _redact_text(value):
    return _URL_RE.sub(lambda match: _redact_argument(match.group(0)), str(value))


def _start_logs(runtime_root, submission_id, carry_from=None):
    global _BOOTSTRAP_LOG, _RUN_BOOTSTRAP_LOG
    previous_global_log = globals().get("_BOOTSTRAP_LOG")
    carry_text = ""
    if carry_from and Path(carry_from).is_file():
        carry_text = Path(carry_from).read_text(encoding="utf-8")
    runtime_root.mkdir(parents=True, exist_ok=True)
    session_path = runtime_root / ".runtime_session_id"
    if session_path.exists():
        session_id = session_path.read_text(encoding="utf-8").strip()
    else:
        session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
        session_path.write_text(session_id + "\n", encoding="utf-8")
    log_root = runtime_root / "logs"
    log_root.mkdir(parents=True, exist_ok=True)
    _BOOTSTRAP_LOG = log_root / f"runtime_{session_id}.log"
    _RUN_BOOTSTRAP_LOG = log_root / f"submission_{submission_id}_{_EXECUTION_ID}.log"
    os.environ["COLAB_RUNTIME_LOG"] = str(_BOOTSTRAP_LOG)
    os.environ["COLAB_SUBMISSION_BOOTSTRAP_LOG"] = str(_RUN_BOOTSTRAP_LOG)
    if carry_text and Path(carry_from) != _RUN_BOOTSTRAP_LOG:
        with _RUN_BOOTSTRAP_LOG.open("a", encoding="utf-8") as stream:
            stream.write(carry_text)
        if previous_global_log and Path(previous_global_log) != _BOOTSTRAP_LOG:
            with _BOOTSTRAP_LOG.open("a", encoding="utf-8") as stream:
                stream.write(carry_text)


def _bootstrap_message(message):
    safe = _redact_text(message)
    print(safe, flush=True)
    for path in (_BOOTSTRAP_LOG, _RUN_BOOTSTRAP_LOG):
        with path.open("a", encoding="utf-8") as stream:
            stream.write(safe + "\n")


def _run_bootstrap_command(command):
    _bootstrap_message("COMMAND START: " + " ".join(_redact_argument(item) for item in command))
    started = time.monotonic()
    process = subprocess.Popen(
        [str(item) for item in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        _bootstrap_message(line.rstrip("\n"))
    return_code = process.wait()
    _bootstrap_message(
        f"COMMAND END: exit_code={return_code} duration_seconds={time.monotonic() - started:.2f}"
    )
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def _read_bootstrap_command(command):
    _bootstrap_message("COMMAND START: " + " ".join(_redact_argument(item) for item in command))
    started = time.monotonic()
    process = subprocess.run(
        [str(item) for item in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    output = process.stdout.rstrip("\n")
    if output:
        for line in output.splitlines():
            _bootstrap_message(line)
    _bootstrap_message(
        f"COMMAND END: exit_code={process.returncode} "
        f"duration_seconds={time.monotonic() - started:.2f}"
    )
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command, output=process.stdout)
    return output


def _project_manifest_files(project_root):
    package_root = project_root / "src" / "project_app"
    files = sorted(
        path
        for path in package_root.rglob("*")
        if path.is_file()
        and not _BUNDLE_EXCLUDED_DIRECTORIES.intersection(
            path.relative_to(package_root).parts
        )
        and path.suffix not in _BUNDLE_EXCLUDED_SUFFIXES
    )
    lock = project_root / "requirements.lock"
    if lock.exists():
        files.append(lock)
    return sorted(files, key=lambda path: str(path.relative_to(project_root)))


def _project_tree_fingerprint(project_root):
    digest = hashlib.sha256()
    for path in _project_manifest_files(project_root):
        digest.update(str(path.relative_to(project_root)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()


def _materialize_embedded_project(configuration):
    if not _EMBEDDED_PROJECT_FILES or not _EMBEDDED_PROJECT_REVISION:
        raise RuntimeError("This source cell has no embedded project bundle; rebuild the notebook")
    runtime_root = Path(configuration["runtime_root"]).expanduser()
    project_root = runtime_root / "embedded_project" / _EMBEDDED_PROJECT_REVISION[:16]
    resolved_project_root = project_root.resolve()
    for relative, encoded in _EMBEDDED_PROJECT_FILES.items():
        relative_path = PurePosixPath(relative)
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise RuntimeError(f"Unsafe embedded project path: {relative}")
        destination = project_root.joinpath(*relative_path.parts)
        try:
            destination.resolve().relative_to(resolved_project_root)
        except ValueError as error:
            raise RuntimeError(f"Embedded project path escapes its isolated root: {relative}") from error
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            content = base64.b64decode(encoded, validate=True)
        except (ValueError, TypeError) as error:
            raise RuntimeError(f"Embedded project file is not valid base64: {relative}") from error
        if not destination.exists() or destination.read_bytes() != content:
            temporary = destination.with_suffix(destination.suffix + ".tmp")
            temporary.write_bytes(content)
            temporary.replace(destination)
    if _project_tree_fingerprint(project_root) != _EMBEDDED_PROJECT_REVISION:
        raise RuntimeError("Embedded project materialization failed its content fingerprint")
    os.environ["COLAB_PROJECT_RESOLVED_REVISION"] = _EMBEDDED_PROJECT_REVISION
    _bootstrap_message(f"Embedded project restored: {_EMBEDDED_PROJECT_REVISION}")
    return project_root


def _restore_local_project(configuration):
    if configuration.get("project_revision") != "local":
        raise RuntimeError("Local source mode requires project_revision='local'")
    project_root = Path(configuration.get("project_source_path", "")).expanduser().resolve()
    if not (project_root / "src" / "project_app").is_dir():
        raise RuntimeError(f"Local project package is missing: {project_root}")
    resolved = "local-" + _project_tree_fingerprint(project_root)
    os.environ["COLAB_PROJECT_RESOLVED_REVISION"] = resolved
    _bootstrap_message(f"Explicit local project reused: {project_root} ({resolved})")
    return project_root


def _normalized_repository_url(value):
    return value.rstrip("/").removesuffix(".git")


def _restore_repository_project(configuration):
    repository_url = configuration.get("project_repository_url", "").strip()
    revision = configuration.get("project_revision", "").strip().lower()
    if not repository_url or not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Repository mode requires a URL and immutable 40-character commit SHA")
    identity = hashlib.sha256((repository_url + "\0" + revision).encode("utf-8")).hexdigest()[:20]
    checkout_root = Path(configuration["runtime_root"]).expanduser() / "project_checkouts" / identity
    if (checkout_root / ".git").is_dir():
        actual_remote = _read_bootstrap_command(
            ["git", "-C", checkout_root, "remote", "get-url", "origin"]
        ).strip()
        if _normalized_repository_url(actual_remote) != _normalized_repository_url(repository_url):
            raise RuntimeError("Existing isolated checkout has an unexpected origin URL")
        actual_head = _read_bootstrap_command(
            ["git", "-C", checkout_root, "rev-parse", "HEAD"]
        ).strip().lower()
        dirty = _read_bootstrap_command(
            ["git", "-C", checkout_root, "status", "--porcelain"]
        ).strip()
        if actual_head == revision and not dirty:
            os.environ["COLAB_PROJECT_RESOLVED_REVISION"] = revision
            _bootstrap_message(f"Pinned repository checkout reused: {revision}")
            return (checkout_root / configuration.get("project_subdirectory", "")).resolve()
    elif checkout_root.exists() and any(checkout_root.iterdir()):
        raise RuntimeError(f"Refusing to replace non-repository checkout: {checkout_root}")
    else:
        checkout_root.parent.mkdir(parents=True, exist_ok=True)
        _run_bootstrap_command(
            ["git", "clone", "--filter=blob:none", "--no-checkout", repository_url, checkout_root]
        )
    _run_bootstrap_command(["git", "-C", checkout_root, "fetch", "--depth", "1", "origin", revision])
    _run_bootstrap_command(["git", "-C", checkout_root, "checkout", "--detach", "--force", revision])
    actual_head = _read_bootstrap_command(
        ["git", "-C", checkout_root, "rev-parse", "HEAD"]
    ).strip().lower()
    if actual_head != revision:
        raise RuntimeError(f"Restored commit {actual_head} does not match submitted commit {revision}")
    dirty = _read_bootstrap_command(
        ["git", "-C", checkout_root, "status", "--porcelain"]
    ).strip()
    if dirty:
        raise RuntimeError("Pinned repository checkout is not clean after restoration")
    project_root = (checkout_root / configuration.get("project_subdirectory", "")).resolve()
    try:
        project_root.relative_to(checkout_root.resolve())
    except ValueError as error:
        raise RuntimeError("project_subdirectory resolves outside the pinned checkout") from error
    if not (project_root / "src" / "project_app").is_dir():
        raise RuntimeError(f"Pinned revision does not contain the project package: {project_root}")
    os.environ["COLAB_PROJECT_RESOLVED_REVISION"] = revision
    _bootstrap_message(f"Pinned repository restored: {_redact_argument(repository_url)}@{revision}")
    return project_root


def _restore_project(snapshot):
    configuration = snapshot["configuration"]
    subdirectory = PurePosixPath(configuration.get("project_subdirectory", "") or ".")
    if subdirectory.is_absolute() or ".." in subdirectory.parts:
        raise RuntimeError("project_subdirectory must be a safe relative path")
    mode = configuration.get("project_source_mode")
    if mode == "embedded":
        return _materialize_embedded_project(configuration)
    if mode == "local":
        return _restore_local_project(configuration)
    if mode == "repository":
        return _restore_repository_project(configuration)
    raise RuntimeError(f"Unsupported project_source_mode: {mode}")


def _mount_drive_if_needed():
    try:
        from google.colab import drive  # type: ignore[import-not-found]
    except ImportError:
        return
    if not Path("/content/drive/MyDrive").is_dir():
        _bootstrap_message("STAGE START: MOUNT_PERSISTENT_STORAGE")
        drive.mount("/content/drive")
        _bootstrap_message("STAGE COMPLETE: MOUNT_PERSISTENT_STORAGE")


def _restore_persistent_pending(runtime_pending):
    _mount_drive_if_needed()
    registry_path = Path("/content/drive/MyDrive/production_colab_registry") / (
        _EMBEDDED_PROJECT_ID + ".json"
    )
    persistent_root = Path(_EMBEDDED_DEFAULT_PERSISTENT_ROOT)
    if registry_path.is_file():
        registry = json.loads(registry_path.read_text(encoding="utf-8"))
        if registry.get("project_id") == _EMBEDDED_PROJECT_ID and registry.get("persistent_root"):
            persistent_root = Path(registry["persistent_root"])
    persistent_pending = persistent_root / "state" / "pending_job.json"
    if not persistent_pending.is_file():
        raise RuntimeError(
            "No frozen job was found in runtime or registered persistent state. "
            "Run the configuration form once, then Run again."
        )
    runtime_pending.parent.mkdir(parents=True, exist_ok=True)
    temporary = runtime_pending.with_suffix(".json.tmp")
    shutil.copy2(persistent_pending, temporary)
    temporary.replace(runtime_pending)
    _bootstrap_message(f"Frozen job restored: {persistent_pending}")
    return json.loads(runtime_pending.read_text(encoding="utf-8"))


def _write_bootstrap_failure(stage, error, snapshot):
    payload = {
        "status": "FAILED",
        "stage": stage,
        "failed_utc": datetime.now(timezone.utc).isoformat(),
        "error": repr(error),
        "traceback": traceback.format_exc(),
        "global_log": str(_BOOTSTRAP_LOG),
        "submission_log": str(_RUN_BOOTSTRAP_LOG),
    }
    if snapshot:
        payload["submission_id"] = snapshot.get("submission_id")
    runtime_failure = _BOOTSTRAP_LOG.parent.parent / "state" / "bootstrap_failure.json"
    try:
        _atomic_write_json(runtime_failure, payload)
    except OSError:
        pass
    if snapshot:
        persistent_root = Path(snapshot.get("configuration", {}).get("persistent_root", ""))
        if persistent_root.exists():
            try:
                _atomic_write_json(persistent_root / "state" / "bootstrap_failure.json", payload)
            except OSError:
                pass


def _execute_submitted_job():
    stage = "LOAD_SUBMITTED_JOB"
    snapshot = None
    pending_default = Path(_EMBEDDED_DEFAULT_RUNTIME_ROOT) / "state" / "pending_job.json"
    pending_path = Path(globals().get("PENDING_JOB_PATH", pending_default))
    try:
        if pending_path.is_file():
            snapshot = json.loads(pending_path.read_text(encoding="utf-8"))
        else:
            _bootstrap_message("No runtime pending job; checking registered persistent state.")
            snapshot = _restore_persistent_pending(pending_path)

        if snapshot.get("configuration", {}).get("project_id") != _EMBEDDED_PROJECT_ID:
            raise RuntimeError(
                "Frozen job project identity does not match this generated notebook"
            )

        configuration = snapshot.get("configuration", {})
        source_mode = configuration.get("project_source_mode")
        submitted_revision = snapshot.get("submitted_project_revision")
        expected_revision = (
            _EMBEDDED_PROJECT_REVISION
            if source_mode == "embedded"
            else configuration.get("project_revision")
        )
        if submitted_revision != expected_revision:
            raise RuntimeError(
                "Frozen job source revision does not match this notebook/project selection"
            )

        recovery_log = _RUN_BOOTSTRAP_LOG
        _start_logs(
            Path(snapshot["configuration"]["runtime_root"]).expanduser(),
            snapshot.get("submission_id", "unknown-submission"),
            carry_from=recovery_log,
        )

        stage = "RESTORE_PROJECT"
        _bootstrap_message("STAGE START: RESTORE_PROJECT")
        project_root = _restore_project(snapshot)
        _bootstrap_message("STAGE COMPLETE: RESTORE_PROJECT")

        source_root = project_root / "src"
        existing = sys.modules.get("project_app")
        if existing is not None and (
            source_mode == "local"
            or not str(getattr(existing, "__file__", "")).startswith(str(source_root))
        ):
            for name in [name for name in sys.modules if name == "project_app" or name.startswith("project_app.")]:
                del sys.modules[name]
        if str(source_root) not in sys.path:
            sys.path.insert(0, str(source_root))

        stage = "VALIDATE_SUBMITTED_JOB"
        from project_app.config import load_pending_job
        from project_app.orchestrator import run_complete_job
        from project_app.results import show_result_panel

        config, validated_snapshot = load_pending_job(pending_path)
        stage = "RUN_COMPLETE_JOB"
        run_result = run_complete_job(
            config,
            project_root,
            submitted_job=validated_snapshot,
        )
        stage = "DELIVER_RESULT"
        delivery_result = show_result_panel(run_result, auto_download=config.auto_download)
        return project_root, config, validated_snapshot, run_result, delivery_result
    except BaseException as exc:
        _bootstrap_message(f"BOOTSTRAP FAILED: stage={stage} error={exc!r}")
        _bootstrap_message(traceback.format_exc())
        _write_bootstrap_failure(stage, exc, snapshot)
        raise


_start_logs(
    Path(os.environ.get("COLAB_RUNTIME_ROOT", _EMBEDDED_DEFAULT_RUNTIME_ROOT)),
    "recovery-"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-"
    + uuid.uuid4().hex[:8],
)
PROJECT_ROOT, CONFIG, SUBMITTED_JOB, RUN_RESULT, DELIVERY_RESULT = _execute_submitted_job()


In [ ]:
#@title Active status, result, and previous runs { display-mode: "form" }
# Requires: only the Python standard library; Drive is mounted on demand in Colab.
# Produces: a stable current/previous-result panel.
# Safe to rerun: yes.

import json
from pathlib import Path

_STATUS_PROJECT_ID = 'lora-converter'  # generated
_STATUS_DEFAULT_PERSISTENT_ROOT = '/content/drive/MyDrive/production_colab/lora-converter'  # generated


def _status_persistent_root():
    try:
        from google.colab import drive  # type: ignore[import-not-found]

        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    except Exception as error:
        print(f"Persistent storage could not be mounted: {error}")
    registry = Path("/content/drive/MyDrive/production_colab_registry") / (
        _STATUS_PROJECT_ID + ".json"
    )
    if registry.is_file():
        try:
            payload = json.loads(registry.read_text(encoding="utf-8"))
            if payload.get("project_id") == _STATUS_PROJECT_ID and payload.get("persistent_root"):
                return Path(payload["persistent_root"])
        except (OSError, ValueError, TypeError) as error:
            print(f"Persistent project registry is unreadable; using the default root: {error}")
    return Path(_STATUS_DEFAULT_PERSISTENT_ROOT)


def _show_file_link(path, label):
    print(f"{label}: {path}")
    if path.is_file():
        try:
            from IPython.display import FileLink, display

            display(FileLink(str(path), result_html_prefix=f"{label}: "))
        except ImportError:
            pass


if "RUN_RESULT" in globals():
    _show_file_link(Path(RUN_RESULT["output_path"]), "Current output")
    _show_file_link(Path(RUN_RESULT["run_log"]), "Current run log")

_persistent_root = _status_persistent_root()
_manifests = sorted((_persistent_root / "runs").glob("*/run_manifest.json"), reverse=True)
if not _manifests:
    print(f"No persistent runs found under {_persistent_root}")
else:
    print("Recent persistent runs")
    for _manifest_path in _manifests[:5]:
        try:
            _manifest = json.loads(_manifest_path.read_text(encoding="utf-8"))
            print(f"- {_manifest.get('run_id', _manifest_path.parent.name)}: {_manifest.get('status', 'UNKNOWN')}")
            _persistent_output = _manifest.get("persistent_output_path")
            if _persistent_output:
                _show_file_link(Path(_persistent_output), "  Persistent output")
            _show_file_link(Path(_manifest.get("run_log", "")), "  Full run log")
        except (OSError, ValueError, TypeError) as _error:
            print(f"- {_manifest_path.parent.name}: UNREADABLE ({_error})")


## Runtime implementation


In [ ]:
#@title Runtime implementation { display-mode: "form" }
# Requires: project source importable from src/.
# Produces: runtime implementation symbols for inspection.
# Safe to rerun: yes; does not prepare the runtime automatically.

from project_app.logging_utils import run_logged_command
from project_app.paths import ProjectPaths
from project_app.runtime import environment_fingerprint, prepare_runtime


## Pipeline implementation


In [ ]:
#@title Pipeline implementation { display-mode: "form" }
# Requires: project source importable from src/.
# Produces: pipeline implementation symbol for inspection.
# Safe to rerun: yes; does not execute the pipeline automatically.

from project_app.pipeline import execute_pipeline


In [ ]:
#@title Cache { display-mode: "form" }
# Requires: project source importable from src/.
# Produces: reusable cache implementation symbols for inspection.
# Safe to rerun: yes; does not read or write a cache automatically.

from project_app.cache import JsonStageCache, cache_key, content_fingerprint


## Result delivery


In [ ]:
#@title Result delivery { display-mode: "form" }
# Requires: optional RUN_RESULT.
# Produces: resolved output path for downstream preview/download UI.
# Safe to rerun: yes.

from project_app.results import show_result_panel

RESULT_PANEL = show_result_panel(RUN_RESULT, auto_download=False) if "RUN_RESULT" in globals() else None


## Maintenance and diagnostics


In [ ]:
#@title Recovery { display-mode: "form" }
print("After an interruption, run the primary Run conversion cell again. Verified downloads and completed conversions are reused.")
print("No files are deleted by this cell. Logs and job records are in the configured Drive folder.")
